## 0 · The Challenge

> **The mission**: Riverside House — a publishing firm with 7 unpublished novels (~197 chapters, 619k words). Confidentiality clause: no manuscript text goes to any public API. Everything runs on a single laptop CPU. The model is GPT-2 medium (355M parameters).

**What we know so far:**

- GPT-2 medium generates coherent English prose.
- **But it doesn't know Riverside's characters, narrative style, or editorial instructions.**

**What's blocking us:**
Base GPT-2 continues any text, but with no domain knowledge. Ask it about "Chapter 7's antagonist" and it generates something plausible-sounding that is entirely wrong. Three gaps: (1) domain knowledge, (2) instruction following, (3) preference alignment with editorial house style.

**What this chapter unlocks:**
Three fine-tuning techniques layered in order — continued pretraining (domain knowledge), SFT (instruction following), DPO (preference alignment) — each addressing exactly one of the three gaps.


# LLM Fine-Tuning Deep Dive, Part 1 of 3: Data-Based Techniques

> **This is Part 1 of a three-notebook fine-tuning arc:**
>
> 1. **Part 1 (this notebook): Data-based techniques** -- what objective/data teaches the
>    behavior (continued pretraining, instruction tuning, preference alignment / DPO).
> 2. [Part 2: Parameter-based techniques + QLoRA & quantization](02-llm-finetuning-parameter-techniques.ipynb)
>    -- how many/which weights are updated (full fine-tuning, partial freezing, LoRA, QLoRA), plus
>    a real, runnable look at post-training quantization for deployment.
> 3. [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) -- head-to-head
>    evaluation of all six trained checkpoints, held-out perplexity, an ablation study, and the final
>    call on what Riverside House actually deploys.
>
> The two axes (data-based and parameter-based) are **independent choices**, which is exactly why
> they split cleanly into separate notebooks -- Part 1 answers "what should the model learn from,"
> Part 2 answers "how much of the model should move while it learns." Every checkpoint Part 1 trains
> is saved to `./checkpoints/` on disk, and Part 2 and Part 3 reload them from there rather than
> assuming this notebook's kernel is still running.

## The brief: Riverside House needs an in-house AI, not an API call

**Riverside House** is a small publishing firm. Everything under [`content/`](content/) is their
**unpublished, proprietary manuscript catalog** -- seven complete novels spanning sci-fi, fantasy,
mystery, historical fiction, cyberpunk, horror, and literary fiction, still under contract, still
unreleased. That's exactly why nobody at Riverside is allowed to paste chapters into a public chatbot
API: the moment draft manuscripts leave the building, the confidentiality clause is broken. Whatever
model they use has to be trained and run **in-house**, on hardware they already own -- a laptop, not a
GPU cluster, and no data ever leaves it.

Riverside's ask has two parts:

1. **An editing assistant** for the ghostwriters and continuity editors -- prompt it with a scene and
   get back a continuation that actually remembers who Aria Voss is and what the Meridian's Promise
   is, follows a direct instruction instead of rambling forever, and reads the way their editors
   _actually_ prefer.
2. **A knowledge base for the rest of the company** -- marketing, licensing, and new hires who need
   answers like "who are the six founding families in the mystery novel?" without reading 197 chapters
   or, worse, guessing.

Today, every one of those people either re-reads old chapters by hand or asks a colleague. That's the
gap this notebook closes -- and by the end we have to pick **one model to actually deploy**, backed by
more than "it read fine to me."

**Goal:** fine-tune a small-but-capable base model (`gpt2-medium`, ~355M parameters) on Riverside's
proprietary catalog -- **seven complete novels** (~619,000 words / ~3.0 MB total, entirely inside
[`content/`](content/)) -- so it learns their characters, invented terminology, and prose style across
genres, demonstrating **every major axis of fine-tuning** along the way:

| Step | Concept                            | Riverside's question                                                                         | Notebook          |
| ---- | ----------------------------------- | ---------------------------------------------------------------------------------------------- | ------------------ |
| 1    | Continued pretraining              | Does it even know our characters and world exist?                                            | Part 1 (this one) |
| 2    | Instruction tuning (SFT)           | Does it follow a "continue this scene" / "answer this question" request instead of rambling? | Part 1 (this one) |
| 3    | Preference alignment (DPO)         | Does it write and answer the way our editors actually prefer?                                | Part 1 (this one) |
| 4    | Full fine-tuning                   | Best quality -- but what does it cost on a laptop?                                           | Part 2            |
| 5    | Partial freezing                   | A cheaper middle ground -- how much quality do we give up?                                   | Part 2            |
| 6    | LoRA + QLoRA                       | The cheapest option -- is it good enough to ship? What if we quantize too?                   | Part 2            |
| 7    | Ablation study                     | What breaks if the deadline forces us to skip a stage?                                       | Part 3            |
| 8    | Head-to-head + held-out perplexity | Which model do we actually deploy in-house?                                                  | Part 3            |

| Axis                | Question it answers                         | Techniques covered here                                                                           |
| -------------------- | -------------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| **Data-based**      | _What objective/data teaches the behavior?_ | Non-instructional (continued pretraining), Instructional (supervised), Preference alignment (DPO) |
| **Parameter-based** | _How many/which weights are updated?_       | Full fine-tuning, Partial (layer freezing), Parameter-efficient (LoRA), QLoRA (Part 2)            |

### What This Notebook Actually Trains

Three distinct models are trained and saved to disk in this notebook. Each one answers a different
question from Riverside's roadmap above:

| #   | Training approach              | Data format                                                                              | Checkpoint                                                                                             |
| --- | ------------------------------- | ------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------- |
| 1   | **Continued pretraining**      | Raw novel paragraphs — no instruction structure, just next-token prediction              | [`./checkpoints/non-instruction-full`](../../../checkpoints/non-instruction-full)                      |
| 2   | **Instruction tuning (SFT)**   | `(instruction prefix + paragraph A, paragraph B)` pairs — prompt is masked from the loss | [`./checkpoints/instruction-lora`](../../../checkpoints/instruction-lora)                              |
| 3   | **Preference alignment (DPO)** | `(prompt, chosen continuation, rejected continuation)` triples                           | [`./checkpoints/preference-dpo`](../../../checkpoints/preference-dpo)                                  |

> The three models **build on each other in order**: #1 is independent; #2 starts from a fresh
> `gpt2-medium` base; #3 continues from #2's trained adapter. All checkpoints land on disk so Part 2
> and Part 3 can reload them without re-running this notebook.

## Corpus (Riverside House's proprietary manuscripts)

- **Location:** [`content/`](content/) -- **7 original, unpublished novels** across diverse genres
  (sci-fi, fantasy, mystery, historical, cyberpunk, horror, literary), totaling **197 chapters**
  (~619,000 words / ~3.0 MB). This directory _is_ Riverside's confidential catalog for the purposes
  of this notebook. See [`content/README.md`](content/README.md) for the full breakdown and
  individual synopses.
- **Genres available:**
  - Sci-fi: _The Weight of Distant Light_ (generation ship, 40 chapters)
  - Fantasy: _The Tidebound Accord_ (epic quest, 33 chapters)
  - Mystery: _The Cartographer's Cipher_ (noir detective, 21 chapters)
  - Historical: _The Silk Merchant's Daughter_ (Tang Dynasty, 23 chapters)
  - Cyberpunk: _Neural Drift_ (memory broker conspiracy, 24 chapters)
  - Horror: _The Hollow Beneath_ (gothic/cosmic, 28 chapters)
  - Literary: _The Weight of Tides_ (marine biology first contact, 28 chapters)
- **Scale note:** Training cells sample a subset (`max_chapters` per genre) by default for fast CPU
  demos. Pass larger limits or `genres=None` to train on the full corpus.

## Setup

Run `setup.ps1` once to create a `.venv` and register the `llm-tuning` Jupyter kernel, then select
that kernel for this notebook.


## Prerequisite Bridge: From Encoder-Decoder Attention to a Decoder-Only Assistant

The transformer foundations introduced three useful shapes: an **encoder** reads an entire input, a **decoder** predicts the next token while respecting a causal mask, and an **encoder-decoder** model lets a decoder attend to an encoded source through cross-attention. Riverside's assistant uses the decoder-only choice: at each turn, the user's instruction, any supplied scene, and the completion form one growing token sequence; causal self-attention lets each new token use everything to its left without seeing its own future.

| Foundation                | Role in this chapter                                   | Why Riverside needs it                                                                               |
| ------------------------- | ------------------------------------------------------ | ---------------------------------------------------------------------------------------------------- |
| Causal decoder            | `gpt2-medium` predicts the next token                  | It can continue prose and answer prompts from one left-to-right context                              |
| Training objective        | Labels say which next tokens should become more likely | Continued pretraining, SFT, and DPO each change what Riverside teaches the same decoder              |
| Encoder / retrieval later | Encodes a query and passages for matching              | It finds current, citable manuscript evidence instead of asking the generator to remember every fact |

So this notebook changes **how a decoder-only model behaves**. It does not turn the model into a dependable catalog lookup system: that next requirement leads to hybrid retrieval after the fine-tuning decision.


## Training Objectives at a Glance

![Three fine-tuning data objectives: continued pretraining, supervised fine-tuning, and direct preference optimization](images/data-objectives-pipeline.png)

The same base model can be adapted by changing both the training data and the objective: continued pretraining learns domain language from raw text, supervised fine-tuning (SFT) learns instruction-following from demonstrations, and DPO learns relative preferences from chosen and rejected responses.


> **Why these specific data × parameter combinations?**
>
> Each concept in this notebook also makes an implicit parameter choice. Those choices weren't
> arbitrary — they follow industry convention:
>
> | Concept                       | Data objective        | Parameter strategy | Why this pairing                                                                                 |
> | ----------------------------- | --------------------- | ------------------ | ------------------------------------------------------------------------------------------------ |
> | 1: Continued pretraining      | Next-token prediction | **Full FT**        | Shows raw weight movement without an adapter layer obscuring it; also the cheapest to understand |
> | 2: Instruction tuning (SFT)   | Instruction-following | **LoRA**           | The industry-standard pairing — almost every production instruction-tuned model uses LoRA        |
> | 3: Preference alignment (DPO) | Human preference      | **LoRA**           | Builds directly on Concept 2's adapter; LoRA-DPO is the de-facto alignment recipe                |
>
> Part 2 then treats the parameter strategy as the _variable_ — using continued pretraining as a
> controlled test bed — so the two axes' costs can be compared directly. The full 3 × 3 combination
> grid (all nine cells, five of them trained) lives in Part 3.


## Table of Contents (Part 1 of 3)

1. [The Brief: Riverside House Needs an In-House AI](#the-brief-riverside-house-needs-an-in-house-ai-not-an-api-call)
   - [Corpus](#corpus-riverside-houses-proprietary-manuscripts)
   - [Setup](#setup)
2. [Why Fine-Tuning? The Three-Gap Problem](#why-fine-tuning-the-three-gap-problem)
3. [Baseline: What Does the Un-Tuned Model Know?](#baseline-what-does-the-un-tuned-model-know)
   - [Setting Up: Which Model, and Why](#setting-up-which-model-and-why)
   - [Choosing a Device](#choosing-a-device-gpu-if-available-cpu-otherwise)
   - [Loading the Tokenizer](#loading-the-tokenizer)
   - [Seeing the Vocabulary in Action](#seeing-the-vocabulary-in-action)
   - [Loading the Base Model](#loading-the-base-model)
   - [A Fixed Test Prompt for Before/After Comparisons](#a-fixed-test-prompt-for-beforeafter-comparisons)
   - [A Reusable `generate()` Helper](#a-reusable-generate-helper)
4. [Test Prompts for Validating Fine-Tuning](#test-prompts)
   - [Understanding One Training Step: A Concrete Example](#understanding-one-training-step-a-concrete-example)
5. [The Fine-Tuning Journey: A Problem-Solution Narrative](#the-fine-tuning-journey-a-problem-solution-narrative)
6. [Concept 1: Continued Pretraining](#concept-1-data-based-non-instructional-fine-tuning-continued-pretraining)
   - [Common Pitfalls: Continued Pretraining](#common-pitfalls-continued-pretraining)
7. [Concept 2: Instruction Tuning (SFT)](#concept-2-data-based-instructional-supervised-fine-tuning)
   - [Common Pitfalls: Instruction Tuning](#common-pitfalls-instruction-tuning)
8. [Concept 3: Preference Alignment (DPO)](#concept-3-data-based-preference-alignment-dpo)
   - [DPO vs. PPO Comparison](#dpo-vs-ppo-at-a-glance)

> Links jump to the matching heading below. If a link doesn't scroll correctly in your Jupyter
> viewer, use `Ctrl+F` / the notebook outline panel with the section title instead -- the numbered
> list above still tells you the order and grouping of everything in this notebook.

**Continues in:**

- [Part 2: Parameter-based techniques + QLoRA & quantization](02-llm-finetuning-parameter-techniques.ipynb)
  -- full fine-tuning, partial freezing, LoRA, QLoRA, and a real post-training quantization demo.
- [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) -- head-to-head
  evaluation, held-out perplexity, ablation study, and the final deployment decision.

---

In [ ]:
from pathlib import Path

# Resolve the notebook's own directory (content/ lives next to this notebook). VS Code's Jupyter
# kernels run with cwd = workspace root, not the notebook's folder, so __vsc_ipynb_file__ (which
# VS Code injects) is the reliable way to find it; __file__ covers plain .py execution.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():

    # Fail loudly instead of guessing another path -- a silent fallback would hide a genuine
    # kernel/cwd misconfiguration instead of surfacing it.
    raise FileNotFoundError(
        f"Could not find the corpus at {CONTENT_DIR}. Open and run this notebook from its own "
        "location in the repo (learning/genai/04-llm/) so its content/ folder resolves correctly."
    )

print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel directory mappings (keys are shorthand aliases, values are actual directory names)
NOVELS = {
    "scifi": "the-weight-of-distant-light",  # 40 chapters
    "fantasy": "the-tidebound-accord",  # 33 chapters
    "mystery": "the-cartographers-cipher",  # 21 chapters
    "historical": "the-silk-merchants-daughter",  # 23 chapters
    "cyberpunk": "neural-drift",  # 24 chapters
    "horror": "the-hollow-beneath",  # 28 chapters
    "literary": "the-weight-of-tides",  # 28 chapters
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from the multi-novel corpus for quick CPU demos.

    Args:
        novels: List of novel aliases to load (e.g., ["scifi", "fantasy"]), or None to
                load all. Available aliases: "scifi", "fantasy", "mystery", "historical",
                "cyberpunk", "horror", "literary" (mapped to directory names).
        max_chapters: Max chapters to load per novel (keeps CPU training fast).
        min_len: Skip paragraphs shorter than this many characters.

    Returns:
        List of paragraph strings from all requested novels.
    """
    if novels is None:
        novels = list(NOVELS.keys())  # load all by default

    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            print(f"Warning: unknown novel alias '{alias}', skipping")
            continue

        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            print(f"Warning: directory {novel_path} not found, skipping")
            continue

        chapter_files = sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]  # cap chapters for fast CPU demos
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):  # split chapter text into paragraph-sized chunks
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:  # skip short, low-signal fragments
                    paragraphs.append(para)

    return paragraphs


# Sample from 4 genres to show multi-genre paragraph diversity
sample_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery", "horror"], max_chapters=2
)
print(f"Loaded {len(sample_paragraphs)} sample paragraphs from 4 novels. First one:\n")
print(sample_paragraphs[20][:421], "...")


## Why Fine-Tuning? The Three-Gap Problem

A pretrained LLM (like GPT-2, LLaMA, or Mistral) has learned language from billions of tokens of web
text, books, and code. This gives it strong **general fluency** and **broad world knowledge**. That's
exactly why `gpt2-medium` is a reasonable starting point for Riverside House's assistant -- it can
already write fluent English. But for Riverside's specific job, it has three concrete gaps:

### Gap 1: Domain Knowledge Gap

**Problem:** The model has never seen Riverside's specific vocabulary, characters, facts, or house
style.

**Example with our corpus:**

- **An editor asks:** `"Who is Aria Voss?"`
- **Base model:** `"Aria Voss is a... [makes up something generic or says 'I don't know']"`
- **After fine-tuning:** `"Aria Voss is the Hold systems technician aboard the Meridian's Promise 
generation ship..."`

**Solution:** Continued pretraining on Riverside's catalog.

---

### Gap 2: Behavior Gap

**Problem:** A raw pretrained model just continues text. It doesn't know how to follow instructions,
answer questions directly, or stop when it should -- which is a problem the moment a ghostwriter wants
to _ask_ something instead of just seeding a paragraph.

**Example:**

- **An editor asks:** `"List the five tides in the Tidebound Accord."`
- **After domain pretraining:** `"List the five tides in the Tidebound Accord. This question has 
puzzled scholars for millennia. Some say there are actually six tides, while others..."` (rambles
  forever)
- **After instruction tuning:** `"The five tides are: water, wind, stone, flame, and void."`

**Solution:** Instruction tuning (supervised fine-tuning) on (prompt, completion) pairs.

---

### Gap 3: Preference Gap

**Problem:** Even an instruction-following model may produce outputs that are technically correct but
not what Riverside's editors actually prefer (too verbose, wrong tone, unhelpful focus) -- and "reads
worse than a human editor would tolerate" is exactly the kind of gap that kills adoption of an internal
tool, even when it "technically works."

**Example:**

- **An editor asks:** `"Who is Aria Voss?"`
- **After instruction tuning:** `"Aria Voss is the systems technician assigned to the Hold of the 
generation ship Meridian's Promise, responsible for monitoring all operational subsystems including 
node seventeen, the Under-Hold maintenance layer, propulsion relay diagnostics..."` (10 more paragraphs
  of technical inventory)
- **After preference alignment:** `"Aria Voss is the Hold's systems tech — she listens to the ship 
the way other people listen to weather."`

**Solution:** Preference alignment (RLHF or DPO) using human preference data.

---

### One Journey, Three Gaps to Close

Which stage to start from -- and how far down this pipeline to go -- depends on which gap is actually
blocking Riverside's assistant today:

```
Pretrained base ---> Continued pretraining ---> Instruction tuning ---> Preference alignment ---> Production model
    (Gap 0)               (Closes Gap 1)              (Closes Gap 2)            (Closes Gap 3)
```

At each stage, you also choose **how many parameters to update** (full fine-tuning vs. freezing vs.
LoRA) -- which, for Riverside House, is really a budget question: they have a laptop, not a GPU
cluster. We'll explore that trade-off after demonstrating the data-based journey.

---

## Baseline: What Does the Un-Tuned Model Know?

Before fine-tuning, let's see what `gpt2-medium` (pretrained on generic web text) produces when
prompted with a scenario from Riverside's sci-fi novel. Since it has never seen this story, expect a
fluent but generic, off-world continuation -- this is the starting point Riverside's editors are stuck
with today.


### Setting Up: Which Model, and Why

Every technique in this notebook needs a base model to fine-tune, so the first thing to pin down is
_which_ pretrained checkpoint we're starting from -- everything downstream (the tokenizer, the
device, every training loop later in the notebook) depends on this one choice.

`MODEL_NAME` is set once, here, and reused everywhere else in the notebook
(`AutoTokenizer.from_pretrained(MODEL_NAME)`, `AutoModelForCausalLM.from_pretrained(MODEL_NAME)`, and
every training cell that loads a fresh copy to fine-tune), so swapping to a bigger or different base
model later only ever means changing this one string.

We pick `gpt2-medium` (~355M params) -- real, pretrained weights, not a toy -- because it's still
small enough to fine-tune on a CPU in a few minutes per stage, which is exactly Riverside's
one-laptop constraint.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2-medium"  # ~355M params, real pretrained weights, CPU-trainable but slow, use heavier models if GPU is available

### Choosing a Device: GPU if Available, CPU Otherwise

`device` tells PyTorch where tensors and model weights should physically live. We need this because
every tensor operation in this notebook (forward pass, backward pass, `.generate()`) has to run on
the same device as the weights, or PyTorch raises a device-mismatch error.

`torch.cuda.is_available()` checks for a usable NVIDIA GPU + CUDA driver; if none is found, we fall
back to `"cpu"` so the notebook still runs end-to-end (just slower) on a laptop with no dedicated
GPU -- Riverside's actual situation.


> **PyTorch → Keras:** `torch.cuda.is_available()` — checks whether a CUDA-capable GPU is visible to PyTorch and returns a bool; the result picks the `device` string (`"cuda"` or `"cpu"`) that every tensor and model call below is pinned to via `.to(device)`. **Keras/TF equivalent:** `tf.config.list_physical_devices('GPU')` — TensorFlow auto-places ops on any visible GPU without needing an explicit device string threaded through the code, so most Keras code skips this check entirely; `tf.device(...)` exists for the rare case you want to force placement.

In [ ]:
device = (
    "cuda" if torch.cuda.is_available() else "cpu"
)  # detects whether a GPU is available
print(f"Using device: {device}")

### Loading the Tokenizer

`gpt2-medium` never sees a single letter of Riverside's manuscripts directly -- it's a stack of
matrix multiplications, and matrices only take numbers. That same constraint applied long before this
notebook existed: to pretrain the model in the first place, billions of tokens of web text had to be
converted into integer IDs using a fixed vocabulary built with byte-pair encoding (BPE), and every
weight in `gpt2-medium`'s embedding matrix and output head was learned against that exact, frozen set
of IDs. Nothing about those weights has changed since -- which is exactly why we can't swap in just
any tokenizer here. Token ID 464 only means whatever `gpt2-medium`'s vocabulary says it means; a
different model's tokenizer would map that same integer to a different, or meaningless, piece of
text -- not a crash, just quietly wrong output. That's why Hugging Face ships every checkpoint paired
with its own tokenizer, and why `AutoTokenizer.from_pretrained(MODEL_NAME)` below is keyed off the
exact same `MODEL_NAME` string used to load the model itself.

One gap remains once that tokenizer is loaded. GPT-2 was pretrained on one continuous stream of
concatenated documents, chopped into fixed-length blocks -- every training example was already
exactly the right length, so padding shorter sequences never came up, and no pad token was ever
defined. This notebook's fine-tuning runs are different: Riverside's chapters get split into
individual paragraphs of wildly different lengths, batched together for speed, and a batched tensor
operation needs every sequence in the batch to share one shape. So right after loading the tokenizer,
we hand it a pad token to borrow -- reusing `eos_token`, the standard convention for GPT-family
models, since it fills the gap without adding a new row to the embedding matrix.


> **PyTorch → Keras:** `AutoTokenizer.from_pretrained(MODEL_NAME)` — downloads and instantiates the exact BPE tokenizer paired with `gpt2-medium`'s pretrained weights, then patches in a pad token since GPT-2 never defined one. **Keras/TF equivalent:** identical call — `AutoTokenizer` is framework-agnostic and pairs equally well with `TFAutoModelForCausalLM`; the divergence between frameworks only starts once the returned ids are fed into a PyTorch vs. TensorFlow model.

In [ ]:
# Load the BPE tokenizer paired with MODEL_NAME's pretrained vocabulary
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:

    # GPT-2 never defined a pad token, so borrow EOS for padding
    tokenizer.pad_token = tokenizer.eos_token


### Seeing the Vocabulary in Action

Before moving on, it's worth actually looking at what `tokenizer` just handed us instead of taking it
on faith. GPT-2's vocabulary is a byte-level BPE (byte-pair encoding) table, and its merges were
learned by counting which byte sequences showed up most often in a huge corpus of ordinary web text.
Crucially, GPT-2's tokenizer treats a leading space as _part of_ the token, not as a separate
character -- so `"signal"` and `" signal"` (with a space in front) are two completely unrelated byte
sequences as far as BPE is concerned, even though they look like "the same word" to a human. Whichever
sequence occurred more often during training earned its own single merged token; the less common one,
if it never came up often enough, stayed split into smaller pieces.

That produces a small surprise below: `signal`, `stared`, and `distant` each split into two pieces
when tokenized alone -- as if they were the very first word of a document, with nothing before them
-- but collapse into a _single_ token the instant they get their leading space back (`Ġsignal`,
`Ġstared`, `Ġdistant`), since `" signal"`-style sequences are everywhere in ordinary prose while the
bare, space-less form is comparatively rare. `Aria` splits into two tokens either way, since even its
common, space-led form (`" Aria"`) wasn't frequent enough in GPT-2's training data to earn a dedicated
token -- it's a name, not an everyday word. (`Ġ` is just how this tokenizer prints "a space came right
before this" when it decodes a token back to a readable string.)


> **PyTorch → Keras:** `tokenizer.encode(word)` / `tokenizer.convert_ids_to_tokens(ids)` — converts raw text to integer token IDs (and back to readable BPE-piece strings) using the framework-agnostic tokenizer loaded above; no tensors are created yet, just plain Python lists. **Keras/TF equivalent:** identical call — `AutoTokenizer` isn't PyTorch- or TF-specific, so a Keras/TF version of this notebook would use this exact same code; only the downstream model call (`TFAutoModelForCausalLM` vs. `AutoModelForCausalLM`) would differ.

In [ ]:
# One example each of a noun, proper noun, verb, and adjective -- all pulled from Riverside's own
# sci-fi opening line, so these are words this notebook already leans on elsewhere.
example_words = {
    "noun": "signal",
    "proper noun": "Aria",
    "verb": "stared",
    "adjective": "distant",
}

for part_of_speech, word in example_words.items():
    ids_alone = tokenizer.encode(word)  # tokenize the word standalone (no leading space)
    ids_mid_sentence = tokenizer.encode(" " + word)  # tokenize as it would appear mid-sentence
    print(f"{part_of_speech.upper()}: {word!r}")
    print(
        f"  as the first word of a text  : ids={ids_alone}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_alone)}"
    )
    print(
        f"  mid-sentence (' {word}')".ljust(31) + f": ids={ids_mid_sentence}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_mid_sentence)}"
    )
    print()

print(
    "'\u0120' at the start of a token marks a leading space -- it's why the same word can tokenize "
    "differently depending on where it appears in a sentence."
)


### Loading the Base Model

`base_model` is the actual pretrained neural network -- 355M real weights downloaded from the
Hugging Face hub, moved onto whichever device we resolved above via `.to(device)`. This untouched
checkpoint is the "before" every fine-tuning technique in this notebook is compared against.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` — downloads gpt2-medium's 355M pretrained weights into a PyTorch `nn.Module` and moves every parameter tensor onto `device` (CPU or GPU) in place. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` — loads the same checkpoint into a `tf.keras.Model` instead; TensorFlow doesn't need an explicit `.to(device)` call since ops are placed on available devices automatically (or via a `tf.device(...)` context).

In [ ]:
# Download gpt2-medium's pretrained weights and move every parameter tensor onto the resolved device
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)


### A Fixed Test Prompt for Before/After Comparisons

`PROMPT` is the one fixed test sentence reused throughout the notebook so "before" vs. "after"
fine-tuning comparisons are always apples-to-apples. It's pulled straight from the sci-fi corpus so
a model that has actually absorbed the catalog has a real chance of continuing it in-world.


In [ ]:
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus

### A Reusable `generate()` Helper

`generate()` is a small wrapper around HuggingFace's `model.generate()` that every later section of
this notebook reuses to compare checkpoints. We need our own wrapper (instead of calling
`model.generate()` directly everywhere) because `model.generate()` returns the prompt tokens _and_
the new tokens concatenated together in one tensor -- there's no built-in "just give me the
continuation" option.

Here's what that actually looks like, using this notebook's own `PROMPT` (which tokenizes to 15
tokens) and a real generation call asking for 15 new tokens:

```text
tokenizer(PROMPT)["input_ids"]          -> 15 tokens          (prompt_len = 15)
model.generate(..., max_new_tokens=15)  -> shape (1, 30)      (15 prompt + 15 new, concatenated)
```

Decoding the full, unsliced tensor prints the prompt right back at you, glued to the front of the
actual answer:

> "Aria Voss stared at the signal counting itself out in prime numbers and began to ponder the
> question, what was it that she had to do?"

`prompt_len` exists so `out[0][prompt_len:]` can drop those first 15 tokens -- without it, every
`print(generate(...))` in this notebook would repeat the whole prompt before showing anything new.

There's a second, smaller wrinkle once only the new tokens are decoded: the same real call above
decodes to `" began to ponder the question, what was it that she had to do?"` -- note the stray
leading space. The first generated token is almost always a space-led token (the Ġ prefix from the
tokenizer-vocabulary demo earlier), so the raw decode nearly always starts with one. `.strip()`
removes it, along with any trailing whitespace/newlines the model happens to generate near the end.


> **PyTorch → Keras:** `model.eval()` / `torch.no_grad()` / `model.generate()` — `.eval()` switches dropout/batchnorm-style layers to inference mode, `torch.no_grad()` disables gradient tracking to save memory during inference, and `.generate()` runs HuggingFace's autoregressive sampling loop (nucleus sampling here via `top_p`/`temperature`). **Keras/TF equivalent:** `TFAutoModelForCausalLM.generate()` — the same HuggingFace `.generate()` API exists on TF models with identical sampling arguments; TF's analog of "eval mode" is passing `training=False` (implicit inside `.generate()`), and there's no separate "no_grad" context since calling a `tf.keras.Model` outside a `GradientTape` block already skips gradient recording.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    """Generate a text continuation for *prompt*.

    Returns **only the newly generated tokens** (prompt is stripped), so every
    print(generate(...)) call in this notebook shows the model's actual output
    without echoing the input back.

    Parameters
    ----------
    model : PreTrainedModel or PeftModel
        Any HuggingFace causal-LM model (base GPT-2, LoRA adapter, DPO policy …)
    prompt : str
        The input text passed to the model.
    max_new_tokens : int
        Hard cap on how many new tokens to generate after the prompt ends.
        The model can stop earlier if it samples the EOS token.

    Notes
    -----
    A real example from this notebook's own `PROMPT` (15 tokens) makes both
    lines concrete. Asking for `max_new_tokens=15` returns `out` with shape
    `(1, 30)` -- the 15 prompt tokens plus 15 new ones, concatenated. Decoding
    all 30 without slicing prints the prompt right back before the answer:

        'Aria Voss stared at the signal counting itself out in prime numbers
         and began to ponder the question, what was it that she had to do?'

    `out[0][prompt_len:]` (`prompt_len = 15` here) drops the first 15 tokens so
    only the new continuation gets decoded. But decoding *just* those 15 new
    tokens gives:

        ' began to ponder the question, what was it that she had to do?'

    -- note the stray leading space: the first generated token is almost
    always a space-led token (the `Ġ` prefix from the tokenizer-vocabulary
    demo above), so the raw decode nearly always starts with one. `.strip()`
    removes it, along with any trailing whitespace/newlines near the end.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(
        device
    )  # use the same tokenizer to tokenize the prompt and convert it to tensor
    prompt_len = inputs["input_ids"].shape[1]  # track where the prompt ends
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # stochastic → varied output
            top_p=0.9,  # nucleus sampling: top 90% mass
            temperature=0.8,  # soften distribution slightly
            pad_token_id=tokenizer.pad_token_id,
        )

    # out[0] shape: (prompt_len + new_tokens,)
    # Slice from prompt_len onward to get ONLY the model's continuation
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return (
        completion
        if completion
        else "[model stopped immediately — sampled EOS as first token]"
    )


print("=== Baseline (no fine-tuning) — model continuation only ===")
print(f"Prompt    : {PROMPT}")
print(f"Completion: {generate(base_model, PROMPT, 20)}")

### Code Walkthrough: Setup Cell

**What just ran — three building blocks used throughout this entire notebook:**

---

**1. `tokenizer.pad_token = tokenizer.eos_token`**

GPT-2 was pretrained on sequences of a fixed length with no padding token in its vocabulary. When the `Trainer` batches examples of different lengths it needs a pad token to fill the shorter sequences. Setting it to `eos_token` (ID 50256) is the standard convention — it tells the tokenizer "treat end-of-sequence as padding." The matching `labels=-100` mask reappears in `tokenize_causal()`, introduced in the Concept 1 section below where the notebook first actually needs to tokenize a training batch.

---

**2. `AutoModelForCausalLM.from_pretrained(MODEL_NAME)`**

This loads the full `gpt2-medium` checkpoint: 355 million parameters, 24 transformer blocks, hidden size 1024. The `.to(device)` call moves all weight tensors to CPU (or GPU if available). The `AutoModel` family is generic — swap `MODEL_NAME` for `"meta-llama/Llama-3.1-8B"` and the rest of the loading code adapts automatically.

---

**3. `generate(model, prompt, max_new_tokens=60)` — decoding strategy**

A thin wrapper around HuggingFace's `model.generate()` with three key choices:

| Parameter         | Value               | Effect                                                   |
| ----------------- | ------------------- | -------------------------------------------------------- |
| `do_sample=True`  | Stochastic decoding | Avoids repetitive, deterministic greedy output           |
| `top_p=0.9`       | Nucleus sampling    | Considers only tokens whose cumulative probability ≥ 90% |
| `temperature=0.8` | Slight smoothing    | Reduces "safe" word dominance without full randomness    |

The function returns **only the newly generated tokens** (after slicing off the prompt), so every `print(generate(...))` call shows the model's actual continuation — not the prompt echoed back at you.

> **PyTorch shape note:** `out = model.generate(...)` returns a tensor of shape `(batch=1, total_len)` where `total_len = prompt_len + max_new_tokens`. We slice `out[0][prompt_len:]` to get just the new tokens.


## Test Prompts

Use these prompts to test whether fine-tuning successfully absorbed domain-specific knowledge from
the seven-novel corpus. A well-tuned model should recognize characters, settings, and continue
narratives in the appropriate style. The baseline model (pretrained only) should produce generic,
off-topic continuations.

### What "success" actually looks like

"Generic" and "on-corpus" are easy to say but vague until you see them side by side. Take the first
sci-fi prompt above, `"Aria Voss checked the Meridian's Promise status panel and"`:

Prompt: `"Aria Voss checked the Meridian's Promise status panel and"`

**Baseline (no fine-tuning) -- what you'll actually see below:**

> A fluent but unrelated continuation -- often a different spaceship, a different job for "Aria," or
> generic technobabble. GPT-2's pretraining corpus has plenty of generic sci-fi, so it always produces
> _something_ readable; it just isn't Riverside's story.

**Expected after continued pretraining on the corpus:**

> A continuation that stays consistent with the real chapter-001 setup: Aria as an engineer who
> monitors the ship "the way other people listened to weather," references to the Under-Hold (the
> ship's off-the-books maintenance underlayer), node seventeen (where she first notices the anomaly),
> or the Lantern (the alien signal the first novel revolves around) -- and a slow-burn, introspective
> tone rather than action-movie beats.

The mystery prompt makes the same point with a different failure mode. Take
`"The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—"`:

**Baseline:** invents a generic, historically-flavored sentence about "founding families" in the
abstract -- it has no idea these are six specific named characters from a land-fraud conspiracy.

**Expected after continued pretraining:** a continuation that stays anchored to the real plot --
the 1879 land fraud in Ashmont Bay, the falsified surveys, Mordecai's murder when he tried to
blackmail the others -- the kind of specific, checkable detail a generic pretrained model has no way
to produce because it was never shown Riverside's unpublished manuscript.

That's the bar every fine-tuning technique in this notebook is measured against: not "does the text
sound plausible" (the baseline already clears that bar), but "does it use the actual names, places,
and plot facts from the corpus." The baseline run right below makes the _before_ half of that
comparison concrete; the Ablation Study near the end of this notebook (Experiment 1) revisits this
exact prompt with a real trained-vs-untrained side-by-side.

### Character & Setting Recognition Tests

**Sci-Fi (The Weight of Distant Light):**

- `"Aria Voss checked the Meridian's Promise status panel and"`
- `"The Keeper's consciousness flickered through node seventeen as"`
- `"In the Under-Hold, Nyla Kade whispered about the prime number signal from"`

**Fantasy (The Tidebound Accord):**

- `"Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as"`
- `"The ancient pillars rose from the Abyssal Rift while Davin Shale"`
- `"The Hollow King's followers, called the Hollowed,"`

**Mystery (The Cartographer's Cipher):**

- `"Elena Voss studied the 1879 survey map and realized the Ashmont Trust"`
- `"Detective Chen examined Adelaide Thorne's body and found the message: 'the foundation must hold'"`
- `"The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—"`

**Historical (The Silk Merchant's Daughter):**

- `"Wei Lian's jade phoenix pendant caught the morning light in Chang'an as"`
- `"Zhang Ming, the jinshi degree holder, wrote in his letter"`
- `"In the Eastern Market, the Wei family silk compound"`

**Cyberpunk (Neural Drift):**

- `"Kai Chen adjusted the neurorig and prepared to extract the memory backup from"`
- `"In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift"`
- `"Victor Tang's consciousness transfer protocol failed when"`

**Horror (The Hollow Beneath):**

- `"Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor"`
- `"The hollow beneath the house breathed, and the entity in the limestone caves"`
- `"Margot found Thaddeus Blackwood's journal warning: never descend past the second chamber"`

**Literary (The Weight of Tides):**

- `"Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about"`
- `"In Willowport, the underwater object near Whitehead Island caused"`
- `"The lobster traps came up bent, and the water temperature dropped fifteen degrees when"`

### Genre Style Continuation Tests

**Sci-Fi narrative momentum:**

- `"Two hundred and fourteen years after the Meridian's Promise left Earth,"`

**Fantasy elemental magic:**

- `"The tide-weavers gathered at Deepwater Crossing as the fifth tide, the void tide,"`

**Mystery noir atmosphere:**

- `"The rain-slicked streets of Ashmont Bay hid secrets from 1879, and Elena Voss"`

**Historical detail & restraint:**

- `"The silk road brought more than trade goods to Tang Dynasty Chang'an—it brought"`

**Cyberpunk tech-noir:**

- `"Memory extraction left traces, neural signatures that couldn't be scrubbed, and Kai Chen"`

**Gothic horror tension:**

- `"The house chose its inhabitants through grief, calling them when they were most vulnerable, and"`

**Literary introspection:**

- `"The ocean held its own memory, deeper and older than human documentation, and Claire"`

### Cross-Novel Vocabulary Tests

These should work across multiple genres if fine-tuning absorbed the corpus style:

- `"The weight of distant"` (tests sci-fi novel phrase bleed)
- `"The tidebound"` (tests fantasy terminology)
- `"permanent removal"` (tests mystery euphemism)
- `"steel in your spine, even if you must hide it beneath"` (tests historical voice)
- `"neural backup"` (tests cyberpunk jargon)
- `"the hollow"` (tests horror atmospheric language)
- `"The water's wrong"` (tests literary marine biology voice)


In [ ]:
# Automated test runner: compare baseline vs fine-tuned on corpus-specific prompts
TEST_PROMPTS = {

    # Sci-fi — The Weight of Distant Light
    "scifi_character": "Aria Voss checked the Meridian's Promise status panel and",
    "scifi_keeper": "The Keeper's consciousness flickered through node seventeen as",

    # Fantasy — The Tidebound Accord
    "fantasy_magic": "Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as",
    "fantasy_hollow_king": "The Hollow King's followers, called the Hollowed, began to gather when",

    # Mystery — The Cartographer's Cipher
    "mystery_conspiracy": "The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—",
    "mystery_elena": "Elena Voss studied the 1879 survey map and realized the Ashmont Trust",

    # Historical — The Silk Merchant's Daughter
    "historical_setting": "Wei Lian's jade phoenix pendant caught the morning light in Chang'an as",
    "historical_silk_road": "The delegation crossed the Taklamakan desert and Wei Lian noted in her ledger",

    # Cyberpunk — Neural Drift
    "cyberpunk_tech": "Kai Chen adjusted the neurorig and prepared to extract the memory backup from",
    "cyberpunk_project": "In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift",

    # Horror — The Hollow Beneath
    "horror_atmosphere": "Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor",
    "horror_chambers": "The tenth chamber of the Hollow pulsed with a light that had no source, and Eleanor",

    # Literary — The Weight of Tides
    "literary_marine": "Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about",
    "literary_contact": "The Observer surfaced near Whitehead Island and Claire understood for the first time that",
}


def test_corpus_knowledge(model, test_prompts=TEST_PROMPTS, max_new_tokens=50):
    """Run all test prompts and return results dict for comparison."""
    results = {}
    prompts = {}
    for key, prompt in test_prompts.items():
        prompts[key] = prompt
        results[key] = generate(model, prompt, max_new_tokens=max_new_tokens)  # this model's continuation for each prompt
    return results, prompts


# Run baseline tests (will show generic, off-corpus continuations)
print(
    "=== BASELINE MODEL (no fine-tuning) - should produce generic continuations ===\n"
)
baseline_results, prompts = test_corpus_knowledge(base_model)
for key, output in baseline_results.items():
    print(f"[{key}]")
    print(prompts[key] + " .... " + output[:200] + "...\n")


Notice the output has no awareness of Aria Voss (from _The Weight of Distant Light_), the _Meridian's
Promise_, the Lantern, or any of the other characters/worlds across the seven novels -- it is fluent
English but a generic, unrelated continuation. This is exactly the gap fine-tuning closes.

Per the "What 'success' actually looks like" example above, a model that had genuinely absorbed this
corpus would instead keep Aria aboard the Meridian's Promise, in her actual role, referencing the
Under-Hold or the Lantern instead of inventing an unrelated ship and crew. This notebook doesn't
re-run `test_corpus_knowledge()` on a fine-tuned checkpoint (fine-tuning a fresh model per test prompt
would multiply the compute cost of every section below), but the Ablation Study near the end trains
and compares checkpoints on a closely related Aria Voss / Meridian's Promise prompt, so you can see
the real before/after side by side.


### Understanding One Training Step: A Concrete Example

Before we start training, let's walk through **exactly what happens** during a single training step of
continued pretraining, using a real paragraph and the real tokenizer -- the code cells further down
run the actual numbers below through `gpt2-medium` instead of making them up.

**Input:** A paragraph from our sci-fi corpus:

> "Aria Voss stared at the signal counting itself out in prime numbers and felt the weight of two
> centuries press against her ribs."

**Step 1: Tokenization**

The tokenizer converts text -> integer IDs, then pads (or truncates) to `max_length=128` so every
example in a batch is the same shape. Concretely, `tokenizer(...)` hands back **two** parallel
128-length arrays for our one-sentence example: `input_ids` (the real token integers, followed by
padding filled with `eos_token`) and `attention_mask` (`1` for every real token, `0` for every padding
slot). That `attention_mask` isn't just bookkeeping -- it's exactly what Step 2 below reads to decide
which of those 128 positions get `-100` in `labels`. Whether a given example ends up with any padding
at all just depends on its own length relative to `max_length=128`: our example here is short, so it
gets padded, and `attention_mask` has trailing `0`s starting exactly where the real text ends; a
paragraph already at or past 128 tokens instead gets truncated down to fit, needs no padding at all,
and so its `attention_mask` comes back all `1`s.

**Step 2: Create Labels for Causal LM -- this is the mask layout**

The label at every position is just the input shifted one to the left: position `i`'s job is to
predict whatever token sits at position `i+1`. The code doesn't build a second, shifted array to do
this, though -- `labels = input_ids.clone()`, with only padding positions overwritten to `-100`, so
`labels` starts out **identical** to `input_ids`. The "shift" happens later, at loss time, by pairing
`logits[:, :-1, :]` (every position's prediction) against `labels[:, 1:]` (the label one slot ahead) --
GPT-2's own `forward()` performs this exact `[:-1]`/`[1:]` alignment automatically the moment you pass
it `labels=...`.

See **"Watching the Shift Happen, Frame by Frame"** below for the animated, position-by-position
walkthrough of exactly how that pairing works -- and contrast it with the instruction-tuning mask
layout later in the notebook, where the **prompt** is masked too, not just the padding.

**Step 3: Forward Pass (Model Prediction)**

Worth pinning down precisely, because it's neither "128 tokens jointly predict a single 129th token"
nor "clone `input_ids` 128 times, masking one more position each time." What actually happens is closer
to the second picture, minus the cloning: GPT-2 doesn't clone anything. Instead, every self-attention
layer has a **fixed causal mask** built in, applied on _every_ forward pass: the attention score from
position `i` to any position `j > i` is zeroed out before the softmax. So position `i`'s logits are
computed using only tokens `0..i` -- the same result you'd get from running the model separately on
each truncated prefix `input_ids[:1]`, `input_ids[:2]`, ..., `input_ids[:128]` and keeping the last
position's prediction each time.

```
Causal mask ( = can attend, · = blocked) for a 4-token example:
        j=0  j=1  j=2  j=3
  i=0:       ·    ·    ·      <- position 0 can only see itself
  i=1:           ·    ·
  i=2:               ·
  i=3:                     <- position 3 sees everything up to itself
```

What one call to `base_model(input_ids=...)` buys you is doing all 128 of those "predict from my own
prefix" computations **in parallel**, as a single batched matrix multiplication, instead of a Python
loop re-running the model 128 times. The causal mask is exactly what makes that legal: since position
`i` is already blocked from attending past itself, computing every position's logits simultaneously
gives the identical result to computing them one at a time with the prefix growing each step -- that's
the actual mechanical reason Step 2's "every position predicts what comes next" works at all, and why a
transformer doesn't need to read tokens one at a time the way an RNN does.

This causal mask is a _different_ mechanism from the `-100` masking in `labels` (Step 2 above): the
causal mask controls what context each position's **prediction** is allowed to use (built into every
attention layer, every forward pass, always on); `-100` controls which positions' predictions the
**loss** bothers to grade (applied once, after the forward pass, only to skip padding). The
`attention_mask` tensor from Step 1 plays a third, narrower role here too -- layered on top of the
fixed causal mask, it additionally blocks every position from attending to _padding_ tokens.

```
Logits shape: (batch_size=1, seq_len=128, vocab_size=50257)
```

These are **raw scores**, not probabilities yet -- one full 50,257-entry vocabulary distribution
(pre-softmax) per position, and position `i`'s distribution only ever depended on tokens `0..i`.

**Step 4: Compute Loss (Cross-Entropy)**

For each real (non-padding) position -- the ones Step 2 did **not** mark `-100` -- we:

1. Convert that position's logits -> probabilities (softmax over all 50,257 vocab entries)
2. Look up the probability the model assigned to the **correct** next token (the label one position
   ahead, from Step 2's shift-by-one pairing)
3. Take the negative log of that probability -- this is the loss for that one position. A confident,
   _correct_ guess (probability near 1) gives a loss near 0; a confident, _wrong_ guess gives a very
   large loss, since `-log(x)` shoots toward infinity as `x` shrinks toward 0.
4. Average the per-position losses across every real token position -- the masked/padding positions
   from Step 2 are skipped entirely via `ignore_index=-100`, exactly the mechanism named there.

**Step 5: Backpropagation**

Compute gradients: `∂Loss/∂W` for every trainable parameter W in the model.

- **Full fine-tuning:** gradients flow to every parameter (~355M for `gpt2-medium`)
- **Partial freezing:** only the unfrozen last few blocks + head get real gradients
- **LoRA:** only the small adapter matrices get gradients (well under 1% of all parameters)

**Step 6: Optimizer Update**

Update weights in the direction that reduces loss:

```
W_new = W_old - learning_rate × gradient
```

**Step 7: Repeat**

Repeated across many steps, these tiny weight nudges accumulate into **learning**: the model becomes
better at predicting tokens that appear in our domain corpus.

---

**Key Takeaways:**


This is the first section that actually plots anything, so this is where we load the visualization
stack -- `matplotlib`/`seaborn` for the charts, `numpy` for the array math behind them. Every later
section that visualizes training internals reuses these same imports.


In [ ]:
# Visualization imports for intuition building
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle
from IPython.display import display, HTML
import warnings

warnings.filterwarnings("ignore")  # suppress noisy library warnings for cleaner notebook output
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})  # sharper plots, readable default font size
sns.set_theme(style="whitegrid", palette="muted")  # consistent seaborn styling across the notebook

print("Visualization libraries loaded.")


### Running Steps 1-2 for Real

The walkthrough above described this in the abstract -- let's actually run it, one step at a time, on
`gpt2-medium` and our example paragraph. First, Steps 1-2: tokenize the sentence into fixed-length
tensors, then build `labels` (a clone of `input_ids`, with padding positions masked to `-100`) exactly
the way described above.


> **PyTorch → Keras:** `input_ids.clone()` / `.to(device)` / boolean-index assignment (`labels[attention_mask == 0] = -100`) — `.clone()` makes an independent copy of a tensor so mutating `labels` won't affect `input_ids`, `.to(device)` moves it to CPU/GPU, and the boolean mask assignment overwrites padding positions in place. **Keras/TF equivalent:** `tf.identity(input_ids)` for the copy — TF tensors are immutable, so masking instead builds a *new* tensor via `tf.where(attention_mask == 0, -100, input_ids)` rather than an in-place assignment; device placement is again automatic instead of an explicit `.to(device)` call.

In [ ]:
import torch.nn.functional as F

example_text = (
    "Aria Voss stared at the signal counting itself out in prime numbers and felt the "
    "weight of two centuries press against her ribs."
)

# Step 1: tokenize
enc = tokenizer(
    example_text,
    truncation=True,
    max_length=128,
    padding="max_length",
    return_tensors="pt",
)
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)
real_len = int(attention_mask.sum().item())  # number of real (non-padding) tokens

# Step 2: build labels -- identical to input_ids, padding positions masked to -100
labels = input_ids.clone()
labels[attention_mask == 0] = -100  # mask padding, exactly like tokenize_causal()

print(
    f"Tokenized to {input_ids.shape[1]} total positions: {real_len} real tokens + "
    f"{input_ids.shape[1] - real_len} padding tokens"
)
print(f"first 8 input_ids : {input_ids[0, :8].tolist()}")
print(
    f"first 8 labels    : {labels[0, :8].tolist()}  <- identical to input_ids (not shifted)"
)
print(
    f"last 8 labels     : {labels[0, -8:].tolist()}  <- all -100 (padding, ignored by the loss)"
)

### Visual Guide: Masking and the Shift, Panel by Panel

Steps 1-2 above are dense in prose -- the figure right below turns them into a picture, built from the
_real_ `input_ids`, `attention_mask`, and `labels` just computed for our example paragraph (nothing
here is a schematic with made-up numbers). Four panels, each isolating one piece of the mechanism:

- **Panel A** -- what `input_ids` actually holds: the real token stream, decoded back into text.
- **Panel B** -- what happens right at the real/padding boundary (position `real_len`): real tokens
  keep their own id as the label; padding positions get overwritten to `-100`.
- **Panel C** -- the shift itself: position `i`'s prediction is graded against the label sitting one
  slot ahead, `label[i+1]` -- the exact mechanism the write-up above described in words only.
- **Panel D** -- the payoff of `-100`: which positions the loss actually counts, and which it silently
  skips via `ignore_index=-100`.


In [ ]:
# Visual guide: masking and the shift, panel by panel -- built from the real tokenizer/model
# output above (input_ids, labels, attention_mask, real_len), not a fabricated schematic.
from matplotlib.patches import Patch


def _clean_tok(token_str):
    """Turn GPT-2 BPE's internal space/newline markers into something readable in a plot."""
    return token_str.replace("\u0120", "\u00b7").replace("\u010a", "\\n")


decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle(
    "Masking and Prediction, Panel by Panel -- real tokens from gpt2-medium's tokenizer",
    fontsize=13,
    fontweight="bold",
)

# Panel A: the real token stream -- what input_ids actually holds
ax_a = axes[0, 0]
n_show_a = 8
for pos in range(n_show_a):
    ax_a.add_patch(
        Rectangle((pos, 0), 0.9, 1, facecolor="lightblue", edgecolor="black")
    )
    ax_a.text(
        pos + 0.45,
        0.5,
        _clean_tok(decoded_tokens[pos]),
        ha="center",
        va="center",
        fontsize=8,
    )
    ax_a.text(
        pos + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray"
    )
ax_a.set_xlim(-0.2, n_show_a + 0.2)
ax_a.set_ylim(-0.6, 1.3)
ax_a.axis("off")
ax_a.set_title(
    "Panel A: input_ids -- the real token stream (position below each box)",
    fontsize=10,
    fontweight="bold",
)
ax_a.legend(
    handles=[Patch(facecolor="lightblue", edgecolor="black", label="Real token")],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18),
    fontsize=7,
)

# Panel B: the real/padding boundary -- where labels actually get masked to -100
ax_b = axes[0, 1]
start_b = max(0, real_len - 5)
end_b = min(input_ids.shape[1], real_len + 4)
window_b = list(range(start_b, end_b))
boundary_j = real_len - start_b
for j, pos in enumerate(window_b):
    is_real = pos < real_len
    color = "mediumseagreen" if is_real else "lightgray"
    label_text = _clean_tok(decoded_tokens[pos]) if is_real else "-100"
    ax_b.add_patch(Rectangle((j, 0), 0.9, 1, facecolor=color, edgecolor="black"))
    ax_b.text(j + 0.45, 0.5, label_text, ha="center", va="center", fontsize=8)
    ax_b.text(
        j + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray"
    )
boundary_line_b = ax_b.axvline(
    boundary_j,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Real/padding boundary",
)
ax_b.set_xlim(-0.2, len(window_b) + 0.2)
ax_b.set_ylim(-0.6, 1.3)
ax_b.axis("off")
ax_b.set_title(
    f"Panel B: labels right at the boundary (real_len={real_len})",
    fontsize=10,
    fontweight="bold",
)
ax_b.legend(
    handles=[
        Patch(
            facecolor="mediumseagreen",
            edgecolor="black",
            label="Real token (label = same token)",
        ),
        Patch(facecolor="lightgray", edgecolor="black", label="Padding (label = -100)"),
        boundary_line_b,
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.3),
    fontsize=7,
)

# Panel C: the shift itself -- prediction at position i graded against label[i+1]
ax_c = axes[1, 0]
n_show_c = 6
for pos in range(n_show_c):
    ax_c.add_patch(
        Rectangle((pos, 0.7), 0.9, 1, facecolor="#ffd9a0", edgecolor="black")
    )
    ax_c.text(
        pos + 0.45,
        1.2,
        _clean_tok(decoded_tokens[pos]),
        ha="center",
        va="center",
        fontsize=8,
    )
    ax_c.text(
        pos + 0.45,
        1.85,
        f"logits[{pos}]",
        ha="center",
        va="center",
        fontsize=6.5,
        color="gray",
    )
    ax_c.add_patch(
        Rectangle((pos, -1.2), 0.9, 1, facecolor="lightblue", edgecolor="black")
    )
    ax_c.text(
        pos + 0.45,
        -0.7,
        _clean_tok(decoded_tokens[pos + 1]),
        ha="center",
        va="center",
        fontsize=8,
    )
    ax_c.text(
        pos + 0.45,
        -1.45,
        f"label[{pos + 1}]",
        ha="center",
        va="center",
        fontsize=6.5,
        color="gray",
    )
    ax_c.annotate(
        "",
        xy=(pos + 0.45, -0.15),
        xytext=(pos + 0.45, 0.65),
        arrowprops=dict(arrowstyle="->", color="darkred", lw=1.5),
    )
ax_c.set_xlim(-0.2, n_show_c + 0.2)
ax_c.set_ylim(-1.7, 2.2)
ax_c.axis("off")
ax_c.set_title(
    "Panel C: the shift -- position i's prediction is graded against label[i+1]",
    fontsize=10,
    fontweight="bold",
)
ax_c.legend(
    handles=[
        Patch(
            facecolor="#ffd9a0",
            edgecolor="black",
            label="Token at position i (what the model has read)",
        ),
        Patch(
            facecolor="lightblue",
            edgecolor="black",
            label="label[i+1] -- what it's graded against",
        ),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.32),
    fontsize=7,
)

# Panel D: the payoff of -100 -- which positions the loss actually counts
ax_d = axes[1, 1]
for j, pos in enumerate(window_b):
    is_real = pos < real_len
    color = "coral" if is_real else "whitesmoke"
    ax_d.add_patch(Rectangle((j, 0), 0.9, 1, facecolor=color, edgecolor="black"))
    if not is_real:
        ax_d.text(
            j + 0.45,
            0.5,
            "skipped",
            ha="center",
            va="center",
            fontsize=7,
            color="gray",
            fontweight="bold",
        )
    ax_d.text(
        j + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray"
    )
boundary_line_d = ax_d.axvline(
    boundary_j,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Real/padding boundary",
)
ax_d.set_xlim(-0.2, len(window_b) + 0.2)
ax_d.set_ylim(-0.6, 1.3)
ax_d.axis("off")
ax_d.set_title(
    "Panel D: ignore_index=-100 in action -- padding contributes zero loss",
    fontsize=10,
    fontweight="bold",
)
ax_d.legend(
    handles=[
        Patch(facecolor="coral", edgecolor="black", label="Counted in the loss"),
        Patch(
            facecolor="whitesmoke",
            edgecolor="black",
            label="Skipped (ignore_index=-100)",
        ),
        boundary_line_d,
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.3),
    fontsize=7,
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(
    f"Real/padding boundary for our example paragraph: position {real_len} out of {input_ids.shape[1]}."
)
print(
    "Panels A-D all use the real tokenizer/model output computed above -- nothing here is fabricated."
)

### Watching the Shift Happen, Frame by Frame

Panel C above shows the shift as a static snapshot of six positions at once -- useful, but it still
asks you to hold "position `i` pairs with label `i+1`" in your head across six boxes simultaneously.
The animation below turns that same idea into a sequence: **one frame per token position**, revealed
one at a time, so the pairing rule shows up as a repeating motion instead of a paragraph of prose.

Watch specifically for two things as the frames advance:

1. The red arrow always points from the top row (position `i`, what the model has just read) down to
   the bottom row **one slot to the right** (label `i+1`) -- never straight down. That one-slot offset
   _is_ the entire "shift" this section has been building up to.
2. The `labels` array itself never rearranges -- every bottom-row token is exactly the same token
   that already sits in `input_ids` at that position. The animation only ever reveals a new _pairing_
   each frame, never a new array.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# One frame per real token position -- reveals the top/bottom pairing one step at a time instead
# of showing all six pairs at once like the static Panel C above.
n_frames_shift = min(10, real_len - 1)

fig_shift, ax_shift = plt.subplots(figsize=(11, 4.5))


def _update_shift(frame):
    ax_shift.clear()
    ax_shift.set_xlim(-0.3, n_frames_shift + 0.3)
    ax_shift.set_ylim(-2.3, 2.3)
    ax_shift.axis("off")

    for pos in range(n_frames_shift):

        # Top row: token AT position `pos` -- what the model has read up through this step
        if pos < frame:
            top_color = "#c9e6c9"  # already graded this pair -- "done"
        elif pos == frame:
            top_color = "#ffb347"  # the pair being graded THIS frame
        else:
            top_color = "whitesmoke"  # not reached yet
        ax_shift.add_patch(
            Rectangle((pos, 0.7), 0.9, 1, facecolor=top_color, edgecolor="black")
        )
        ax_shift.text(
            pos + 0.45,
            1.2,
            _clean_tok(decoded_tokens[pos]),
            ha="center",
            va="center",
            fontsize=9,
        )
        ax_shift.text(
            pos + 0.45,
            1.85,
            f"pos {pos}",
            ha="center",
            va="center",
            fontsize=7,
            color="gray",
        )

        # Bottom row: label[pos + 1] -- only revealed once its pair has actually been reached
        if pos <= frame:
            bottom_color = "#c9e6c9" if pos < frame else "#8ecae6"
            ax_shift.add_patch(
                Rectangle(
                    (pos, -1.7), 0.9, 1, facecolor=bottom_color, edgecolor="black"
                )
            )
            ax_shift.text(
                pos + 0.45,
                -1.2,
                _clean_tok(decoded_tokens[pos + 1]),
                ha="center",
                va="center",
                fontsize=9,
            )
            ax_shift.text(
                pos + 0.45,
                -1.95,
                f"label[{pos + 1}]",
                ha="center",
                va="center",
                fontsize=7,
                color="gray",
            )
            ax_shift.annotate(
                "",
                xy=(pos + 0.45, -0.65),
                xytext=(pos + 0.45, 0.65),
                arrowprops=dict(
                    arrowstyle="->",
                    color="darkred" if pos == frame else "#9aa5b1",
                    lw=2.2 if pos == frame else 1,
                ),
            )
        else:
            ax_shift.add_patch(
                Rectangle(
                    (pos, -1.7), 0.9, 1, facecolor="whitesmoke", edgecolor="black"
                )
            )
            ax_shift.text(
                pos + 0.45,
                -1.2,
                "?",
                ha="center",
                va="center",
                fontsize=9,
                color="lightgray",
            )

    current_tok = _clean_tok(decoded_tokens[frame])
    label_tok = _clean_tok(decoded_tokens[frame + 1])
    ax_shift.set_title(
        f"Position {frame}: model has read through '{current_tok}' \u2192 prediction here is graded "
        f"against label[{frame + 1}] = '{label_tok}'\n(same labels array throughout -- only the "
        f"pairing shown by the arrow shifts)",
        fontsize=10,
        fontweight="bold",
    )
    return []


anim_shift = FuncAnimation(
    fig_shift, _update_shift, frames=n_frames_shift, interval=900, blit=False
)
plt.close(fig_shift)  # prevent a duplicate static frame from also rendering

print(
    "Animation: one frame per token position. Top row = what the model has read so far; bottom row = "
    "the label it's graded against for that position's prediction. The red arrow always points from "
    "position i (top) down to label[i+1] (bottom, one slot over) -- that one-slot offset is the whole "
    "'shift'. Green = already-graded pairs; gray = not reached yet.\n"
)
display(HTML(anim_shift.to_jshtml(fps=2)))

### Steps 3-5: Forward Pass, Loss, and Backprop in One Call

Passing `labels=...` into `base_model(...)` makes HuggingFace do Steps 3 and 4 internally in a single
call: it runs the forward pass (producing `outputs.logits`), then shifts and compares logits against
labels the way described above, returning the averaged result as `outputs.loss`. `step_loss.backward()`
is Step 5 -- PyTorch's autograd walks backward through every operation that produced `step_loss` and
computes `∂Loss/∂W` for every parameter that needs a gradient, without us deriving any calculus by hand.
`base_model.train()` beforehand just tells dropout-style layers to behave in "training mode" for this
one pass; it's switched back to `.eval()` a couple of cells down, once this illustrative pass is done,
so nothing here leaks into the rest of the notebook.


> **PyTorch → Keras:** `model.train()` / `model.zero_grad()` / `model(..., labels=...)` / `loss.backward()` — `.train()` re-enables dropout for this illustrative pass, `.zero_grad()` clears stale gradients from any previous backward call, passing `labels=` makes the HuggingFace model compute cross-entropy internally and return it as `outputs.loss`, and `.backward()` triggers PyTorch autograd to populate `.grad` on every parameter. **Keras/TF equivalent:** `tf.GradientTape()` — Keras/TF has no separate "training mode" flag on the model itself (`training=True` is passed as a call argument instead) and no manual `.backward()`; you'd wrap the forward pass in `with tf.GradientTape() as tape:`, then call `tape.gradient(loss, model.trainable_variables)` to get the equivalent of populated `.grad` attributes.

In [ ]:
base_model.train()  # need gradients for this one illustrative pass; restored to eval() below
base_model.zero_grad()

# GPT-2 defaults to the SDPA attention backend, which doesn't return per-position attention weights
# (output_attentions=True would otherwise come back as an empty tuple). Switch to eager just for this
# one illustrative pass -- mathematically identical, just slower -- and switch back right after.
_prev_attn_impl = base_model.config._attn_implementation
base_model.set_attn_implementation("eager")
outputs = base_model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    output_attentions=True,  # keeps the per-layer attention weights for the animation below
)
base_model.set_attn_implementation(_prev_attn_impl)  # restore the original (SDPA) attention backend
step_loss = outputs.loss
step_loss.backward()  # compute gradients for every parameter via backprop

print(
    f"logits shape: {tuple(outputs.logits.shape)}  (one 50,257-vocab prediction per position)"
)
print(f"average loss across all real positions: {step_loss.item():.3f}")


### Watching Parallel Attention, One Matmul Instead of Four Passes

The walkthrough above explained _why_ one call to `base_model(...)` can stand in for many sequential
passes: the causal mask blocks position `i` from ever attending to `j > i`, so every row's answer is
already independent of the rows below it. The animation below makes that concrete on a 4-token window
sliced from the front of `example_text`, using the **real attention weights** `outputs.attentions`
just returned by the Step 3 forward pass above (layer 0, head 0) -- not fabricated numbers.

Left panel: what four _sequential_ passes would look like if attention had to arrive one row at a
time, the way an RNN reads left to right. Right panel: what a real forward pass actually returns --
every row of the causal-masked attention matrix already exists after a single matrix multiplication.
The only thing "revealed frame by frame" here is the animation itself; inside the model, all four
rows are already there from frame one.


> **PyTorch → Keras:** `outputs.attentions[0][0, 0, :window_len, :window_len].detach().cpu().numpy()` — `.detach()` removes a tensor from the autograd graph so no gradient bookkeeping follows it, `.cpu()` moves it off the GPU if it was there, and `.numpy()` converts it to a plain NumPy array for matplotlib to plot. **Keras/TF equivalent:** `tensor.numpy()` — TF's eager-mode tensors expose `.numpy()` directly with no separate detach/cpu step needed, since tensors returned outside a `GradientTape` already carry no gradient history and `.numpy()` implicitly copies off-device if necessary.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Real attention weights (layer 0, head 0) for a 4-token window at the front of our example
# paragraph -- pulled straight from outputs.attentions returned by the Step 3 forward pass above,
# not fabricated. Row i = how much query position i attends to each key position j <= i.
window_len = 4
window_tokens = [_clean_tok(t) for t in decoded_tokens[:window_len]]
attn_window = (
    outputs.attentions[0][0, 0, :window_len, :window_len].detach().cpu().numpy()
)
vmax_attn = attn_window.max()

fig_attn, (ax_seq, ax_par) = plt.subplots(1, 2, figsize=(11, 4.6))


def _row_grid(up_to_row):
    """Rows 0..up_to_row filled with real attention weights (causal-masked); later rows blank."""
    grid = np.full((window_len, window_len), np.nan)
    for i in range(window_len):
        if i <= up_to_row:
            grid[i, : i + 1] = attn_window[i, : i + 1]
    return grid


def _draw_attn(ax, grid, title):
    ax.clear()
    ax.imshow(grid, cmap="viridis", vmin=0, vmax=vmax_attn, aspect="equal")
    ax.set_xticks(range(window_len))
    ax.set_yticks(range(window_len))
    ax.set_xticklabels(window_tokens, fontsize=8)
    ax.set_yticklabels(window_tokens, fontsize=8)
    ax.set_xlabel("key position j", fontsize=8)
    ax.set_ylabel("query position i", fontsize=8)
    ax.set_title(title, fontsize=9, fontweight="bold")
    for i in range(window_len):
        for j in range(window_len):
            if not np.isnan(grid[i, j]):
                ax.text(
                    j,
                    i,
                    f"{grid[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if grid[i, j] < vmax_attn * 0.6 else "black",
                )


n_frames_attn = window_len + 1  # one extra hold frame at the end


def _update_attn(frame):
    seq_row = min(frame, window_len - 1)
    _draw_attn(
        ax_seq,
        _row_grid(seq_row),
        f"If attention ran sequentially\n(row {seq_row} just 'arrived')",
    )
    _draw_attn(
        ax_par,
        _row_grid(window_len - 1),
        "What one forward pass actually returns\n(all rows already computed, one matmul)",
    )
    fig_attn.suptitle(
        f"Frame {frame + 1}/{n_frames_attn} -- real layer-0/head-0 attention, first {window_len} tokens",
        fontsize=10,
    )
    return []


anim_attn = FuncAnimation(
    fig_attn, _update_attn, frames=n_frames_attn, interval=900, blit=False
)
plt.close(fig_attn)  # prevent a duplicate static frame from also rendering

print(
    "Animation: the same real layer-0/head-0 attention weights on both sides. Left panel pretends "
    "attention arrives one row per step, like an RNN reading left to right. Right panel shows what "
    "base_model(...) actually returns -- every row already filled in after a single parallel matmul, "
    "because the causal mask (blocked/blank cells) already guarantees row i never depended on the "
    "rows below it.\n"
)
display(HTML(anim_attn.to_jshtml(fps=2)))

### Zooming Into the Loss: Per-Token Detail

`step_loss` above is already the _average_ loss HuggingFace computed internally -- useful for training,
but it hides the position-by-position detail Step 4 actually describes. This cell recomputes that same
cross-entropy manually, one position at a time, using the exact `[:-1]` / `[1:]` alignment from
earlier, so we can see individual token losses instead of one averaged number.


> **PyTorch → Keras:** `F.cross_entropy(shift_logits, shift_labels, reduction="none", ignore_index=-100)` — computes per-position cross-entropy loss manually (instead of the averaged loss HuggingFace returns via `labels=`), with `reduction="none"` keeping one loss value per token and `ignore_index=-100` skipping masked positions entirely. **Keras/TF equivalent:** `tf.keras.losses.SparseCategoricalCrossentropy(reduction='none')` — TF/Keras has no built-in `ignore_index`; masking is instead done by multiplying the per-token loss by a `0`/`1` mask tensor (or passing a matching `sample_weight`) built from the same padding/prompt logic.

In [ ]:
# Real per-position loss for the first few real (non-padding) tokens
shift_logits = outputs.logits[0, :-1, :]  # drop the last position's logits (nothing left to predict)
shift_labels = labels[0, 1:]  # drop the first label so index i lines up with logits[i]'s target
per_token_loss = F.cross_entropy(
    shift_logits, shift_labels, reduction="none", ignore_index=-100
)  # unreduced, per-position loss so individual positions can be inspected
positions_to_show = min(8, real_len - 1)
losses_per_pos = per_token_loss[:positions_to_show].detach().cpu().numpy()  # move to numpy for printing

print(f"Per-token loss for the first {positions_to_show} real positions:")
print(losses_per_pos.round(3))
print(
    f"Mean of these {positions_to_show}: {losses_per_pos.mean():.3f}  "
    f"(compare to the full-sequence average loss printed above: {step_loss.item():.3f})"
)


### Which Weights Actually Move?

`gpt2-medium` is made of a few distinct kinds of weights (embeddings, attention, MLP, layernorms,
output head), and the three techniques in this notebook don't touch the same ones:

| Component                       | Full Fine-Tuning             | Partial Freezing                          | LoRA                                |
| ------------------------------- | ---------------------------- | ----------------------------------------- | ----------------------------------- |
| Token + position embeddings     | Updated                      | Frozen                                    | Frozen                              |
| Attention (Q/K/V + output proj) | Updated                      | Frozen (early blocks), updated (last few) | Frozen base + small adapter updated |
| MLP / FFN                       | Updated                      | Frozen (early blocks), updated (last few) | Frozen                              |
| LayerNorms                      | Updated                      | Frozen (early blocks), updated (last few) | Frozen                              |
| Output head                     | Updated (tied to embeddings) | Updated                                   | Frozen                              |

`step_loss.backward()` above already populated a real `.grad` on every weight, since this example runs
full fine-tuning (nothing frozen yet) -- exactly the gradient the backprop animation below traces
block by block.


### Watching Backprop Flow, Block by Block

Step 5 says "compute gradients for every parameter," but that's not something that happens all at
once, or in forward order. Backprop walks the computation graph in **reverse**: the chain rule means
block 23's gradient (the one right next to the loss) can be computed immediately, but block 22's
gradient needs block 23's result first, block 21's needs block 22's, and so on -- all the way back to
block 0, next to the embeddings. That's the entire reason it's called "**back**"-propagation.

The animation below computes the real per-block gradient norms directly from `base_model`'s populated
`.grad` tensors, then reveals them one block at a time, in that same reverse order -- block 23 lights
up first, block 0 lights up last.


> **PyTorch → Keras:** `model.named_parameters()` / `p.grad.norm()` / `torch.norm(torch.stack([...]))` — `named_parameters()` iterates every learnable tensor with its dotted name (used here to filter by transformer block index), `.grad` holds the gradient populated by the earlier `.backward()` call, and `.norm()` computes its L2 magnitude; `torch.stack` + `torch.norm` combine several per-parameter norms into one per-block value. **Keras/TF equivalent:** `model.trainable_variables` — TF/Keras exposes trainable weights as a flat list (with `.name` as the equivalent of parameter names) rather than storing gradients on the tensor itself; gradients instead come back as a separate list from `tape.gradient(...)`, and `tf.norm(tf.stack([...]))` is the direct equivalent of the norm/stack combination here.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Real gradient magnitude per transformer block (shows actual gradient flow, not a fabricated curve)
n_blocks = base_model.config.n_layer
block_grad_norms = []
for i in range(n_blocks):
    block_params = [
        p
        for n, p in base_model.named_parameters()
        if f"h.{i}." in n and p.grad is not None
    ]
    norm = (
        torch.norm(torch.stack([p.grad.norm() for p in block_params])).item()
        if block_params
        else 0.0
    )
    block_grad_norms.append(norm)

# One frame per block, revealed in REVERSE order (block 23 first, block 0 last) -- animated in the
# order backprop actually visits them.
block_order = list(range(n_blocks - 1, -1, -1))  # 23, 22, ..., 0

fig_bp, ax_bp = plt.subplots(figsize=(9, 6))


def _update_backprop(frame):
    ax_bp.clear()
    current_block = block_order[frame]
    revealed = set(block_order[: frame + 1])
    heights = [block_grad_norms[i] if i in revealed else 0.0 for i in range(n_blocks)]
    colors = [
        (
            "#d62728"
            if i == current_block
            else ("#9467bd" if i in revealed else "whitesmoke")
        )
        for i in range(n_blocks)
    ]
    ax_bp.barh(np.arange(n_blocks), heights, color=colors)
    ax_bp.set_xlim(0, max(block_grad_norms) * 1.15)
    ax_bp.set_ylim(-0.5, n_blocks - 0.5)
    ax_bp.invert_yaxis()
    ax_bp.set_xlabel("Real gradient norm", fontsize=9)
    ax_bp.set_ylabel(
        "Block (0 = nearest embeddings, 23 = nearest the loss)", fontsize=8
    )
    ax_bp.set_title(
        f"Backprop step {frame + 1}/{n_blocks}: gradient just reached block {current_block}",
        fontsize=10,
        fontweight="bold",
    )


anim_backprop = FuncAnimation(
    fig_bp, _update_backprop, frames=n_blocks, interval=180, blit=False
)
plt.close(fig_bp)  # prevent a duplicate static frame from also rendering

print(
    "Animation: one frame per transformer block, revealed in REVERSE order -- block 23 (nearest the "
    "loss) lights up first, block 0 (nearest the embeddings) lights up last. Red = the block whose "
    "gradient just arrived; purple = already computed; gray = not reached yet. That reverse ordering "
    "is the entire meaning of 'back'-propagation: each block's gradient formula needs the block AFTER "
    "it, already computed, so the signal has to travel backward through the model.\n"
)
display(HTML(anim_backprop.to_jshtml(fps=5)))

### Step 6: The Actual Weight Update

This is the step every fine-tuning technique in this notebook is really about:
`W_new = W_old - learning_rate × gradient`. We're not letting an optimizer do this at scale yet (that
happens inside `Trainer.train()` in the very next section) -- instead, we manually apply that same
formula to one real weight from `base_model`, so the update is visible instead of buried inside
thousands of simultaneous parameter updates. One nudge is tiny: `lr=5e-5` times a small gradient often
works out to around `1e-8`, far too small to notice at 6 decimal places, which is why the printed delta
below uses scientific notation. `base_model.zero_grad()` and `.eval()` afterward reset the model back
to exactly how the rest of the notebook expects to find it -- this was a one-off illustration, not a
real training step, so nothing here is meant to persist. Once that's done, let's put all six steps
into one figure below.


> **PyTorch → Keras:** `sample_param.data` / `sample_param.grad` — `.data` accesses a parameter's raw tensor values while bypassing autograd tracking (used here purely to read/print a value), and `.grad` reads the gradient populated by the earlier backward pass; the manual `new_weight = old_weight + weight_delta` line reproduces the SGD update rule `W_new = W_old - lr × gradient` by hand instead of calling an optimizer. **Keras/TF equivalent:** `variable.numpy()` / `tape.gradient(...)` — a Keras/TF version would read `variable.numpy()` for the raw value and the corresponding entry from `tape.gradient(loss, model.trainable_variables)` for the gradient, then apply the same formula manually, or just call `optimizer.apply_gradients(...)` for the real (non-illustrative) update.

In [ ]:
# Real weight update on one real parameter, using the actual computed gradient.
# LR=5e-5 times a small gradient is often ~1e-8 -- too small to see at 6 decimal places, so we
# print W_old/W_new AND the delta in scientific notation so the update is actually visible.
sample_name, sample_param = next(
    (n, p)
    for n, p in base_model.named_parameters()
    if p.grad is not None and p.dim() == 2
)  # first 2D parameter with a real gradient from the backward pass above
old_weight = sample_param.data.flatten()[0].item()  # one real scalar weight, before the update
sample_grad = sample_param.grad.flatten()[0].item()  # that same weight's real gradient
lr = 5e-5
weight_delta = -lr * sample_grad  # the actual gradient-descent update rule
new_weight = old_weight + weight_delta

base_model.zero_grad()
base_model.eval()  # leave base_model exactly as it was for the rest of the notebook

print(f"Parameter: {sample_name}")
print(f"W_old = {old_weight:.6f}   gradient = {sample_grad:.3e}   LR = {lr:.0e}")
print(f"\u0394W = -lr * gradient = {weight_delta:+.3e}   ->  W_new = {new_weight:.6f}")


### Watching the Optimizer Nudge, Zoomed In

Step 6's formula, `W_new = W_old - lr × gradient`, produced a real number above -- but at 6 decimal
places the change was invisible, which is exactly why the cell above had to print it in scientific
notation. The animation below plots those same `W_old`/`W_new` values, zoomed into a window just
barely wide enough to fit both, so the real (tiny) movement becomes visible instead of implied. Nothing
here is exaggerated or fabricated -- only zoomed in.

> **What `W_old` and `W_new` actually are here**
>
> These are **a single floating-point number** — specifically, the `.flatten()[0]` element (the very
> first weight in the matrix when laid out in row-major order) of the **first 2D parameter that had
> a non-null gradient** (found by `next(...)` in the cell above, whose name is printed as
> `"Parameter: ..."` when that cell runs). That one scalar was chosen purely to make the update
> _visible_; it is not a per-block average, a norm, or any kind of aggregate.
>
> The same `W_new = W_old - lr × gradient` formula is simultaneously applied to every one of
> `gpt2-medium`'s ~355M individual weights during a real optimizer step — this animation just zooms
> in on one of them so the mechanics are tangible. The per-block weight-delta chart further down in
> this notebook shows the _spread_ of those simultaneous updates across all 24 transformer blocks.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Animate the real W_old -> W_new nudge from the cell above. The axis is zoomed tightly around the
# two real values (not an arbitrary exaggeration) so the genuinely tiny movement is actually visible.
n_frames_update = 20
pad = abs(weight_delta) * 1.5 if weight_delta != 0 else 1e-9
axis_lo, axis_hi = min(old_weight, new_weight) - pad, max(old_weight, new_weight) + pad

fig_update, ax_update = plt.subplots(figsize=(10, 3))


def _update_weight_anim(frame):
    ax_update.clear()
    t = frame / (n_frames_update - 1)
    current_value = old_weight + t * weight_delta
    ax_update.axvline(
        old_weight, color="steelblue", linestyle="--", linewidth=1.5, label="W_old"
    )
    ax_update.axvline(
        new_weight, color="green", linestyle="--", linewidth=1.5, label="W_new"
    )
    ax_update.plot(
        [current_value], [0], marker="o", markersize=14, color="darkred", zorder=5
    )
    ax_update.set_xlim(axis_lo, axis_hi)
    ax_update.set_ylim(-1, 1)
    ax_update.set_yticks([])
    ax_update.set_xlabel(
        f"Weight value, zoomed to the real W_old/W_new window (parameter: {sample_name})",
        fontsize=8,
    )
    ax_update.legend(loc="upper left", fontsize=8)
    ax_update.set_title(
        f"Step 6: applying \u0394W = -lr \u00d7 gradient -- {t:.0%} of the way there\n"
        f"current = {current_value:.10f}  (W_old={old_weight:.10f}, W_new={new_weight:.10f})",
        fontsize=9,
        fontweight="bold",
    )


anim_update = FuncAnimation(
    fig_update, _update_weight_anim, frames=n_frames_update, interval=120, blit=False
)
plt.close(fig_update)  # prevent a duplicate static frame from also rendering

print(
    f"Animation: the red dot slides from W_old to W_new -- the same real numbers printed above "
    f"(\u0394W = {weight_delta:+.3e}), plotted on an axis zoomed tightly around the two values so the "
    f"movement is actually visible. At normal scale this nudge is far too small to see, which is "
    f"exactly why fine-tuning needs thousands of steps, not one, to add up to real learning.\n"
)
display(HTML(anim_update.to_jshtml(fps=8)))

In [ ]:
# Anatomy of one training step -- all six real numbers above, laid out in one figure
# (not fabricated numbers: every value below came from the cells above, computed for real on base_model)
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Anatomy of One Training Step (Continued Pretraining) -- real numbers from gpt2-medium",
    fontsize=13,
    fontweight="bold",
)

# Step 1: Tokenization
ax1 = axes[0, 0]
ax1.text(
    0.5,
    0.88,
    "Step 1: Tokenization",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax1.transAxes,
)
ax1.text(
    0.5,
    0.45,
    f'"{example_text[:38]}..."\n\u2193\n{input_ids[0, :8].tolist()} ...',
    ha="center",
    va="center",
    fontsize=9,
    transform=ax1.transAxes,
    bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
)
ax1.axis("off")
ax1.legend(
    handles=[
        Patch(
            facecolor="lightblue",
            alpha=0.7,
            edgecolor="black",
            label="Text -> token ids",
        )
    ],
    loc="lower center",
    fontsize=7,
)

# Step 2: The mask layout (real tokens vs. padding) -- the actual answer to "how is the mask laid out"
ax2 = axes[0, 1]
mask_row = attention_mask[0].cpu().numpy().reshape(1, -1)
ax2.imshow(
    mask_row, cmap="Greens", aspect="auto", vmin=0, vmax=1, extent=[0, 128, 0, 1]
)
boundary_line = ax2.axvline(
    real_len, color="red", linestyle="--", linewidth=1.5, label="Real/padding boundary"
)
ax2.set_yticks([])
ax2.set_xlabel("Token position (0-128)", fontsize=8)
ax2.set_title(
    f"Step 2: Mask Layout\n{real_len} real tokens (green, active) +\n"
    f"{128 - real_len} padding (white, labels=-100)",
    fontsize=9,
    fontweight="bold",
)
ax2.legend(
    handles=[
        Patch(facecolor="darkgreen", label="Real token (active)"),
        Patch(facecolor="white", edgecolor="black", label="Padding (labels=-100)"),
        boundary_line,
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=1,
    fontsize=7,
)

# Step 3: Forward pass -- NOT the raw logit values (those are just arbitrary numbers to look at).
# What actually matters for "forward pass" is the causal mask that lets one parallel call stand in
# for 128 sequential ones: position i's row can only see columns j <= i.
ax3 = axes[0, 2]
mask_window = min(16, real_len)  # small enough that individual cells are still readable
causal_mask_matrix = np.tril(
    np.ones((mask_window, mask_window))
)  # 1 = i can attend to j
ax3.imshow(causal_mask_matrix, cmap="Greens", vmin=0, vmax=1, aspect="equal")
ax3.set_xlabel("Position j (key)", fontsize=8)
ax3.set_ylabel("Position i (query)", fontsize=8)
ax3.set_title(
    f"Step 3: Forward Pass\ncausal mask, first {mask_window} positions",
    fontsize=9,
    fontweight="bold",
)
ax3.set_xticks(range(0, mask_window, max(1, mask_window // 4)))
ax3.set_yticks(range(0, mask_window, max(1, mask_window // 4)))
ax3.legend(
    handles=[
        Patch(facecolor="darkgreen", label="i can attend to j (j \u2264 i)"),
        Patch(
            facecolor="white", edgecolor="black", label="blocked (j > i, the future)"
        ),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    fontsize=6.5,
)

# Step 4: Real per-position loss
ax4 = axes[1, 0]
positions = np.arange(len(losses_per_pos))
ax4.bar(
    positions,
    losses_per_pos,
    color="coral",
    alpha=0.8,
    width=0.6,
    label="Per-token loss",
)
ax4.axhline(
    losses_per_pos.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Mean (shown)",
)
ax4.set_xlabel("Token position", fontsize=9)
ax4.set_ylabel("Loss", fontsize=9)
ax4.set_title(
    f"Step 4: Compute Loss\nfull-sequence avg = {step_loss.item():.3f}",
    fontsize=9,
    fontweight="bold",
)
ax4.legend(fontsize=8)

# Step 5: Real gradient magnitude per block
ax5 = axes[1, 1]
tick_stride = max(1, n_blocks // 8)
ax5.barh(
    np.arange(n_blocks),
    block_grad_norms,
    color="purple",
    alpha=0.7,
    label="Gradient norm per block",
)
ax5.set_yticks(np.arange(0, n_blocks, tick_stride))
ax5.set_yticklabels([f"Block {i}" for i in range(0, n_blocks, tick_stride)], fontsize=8)
ax5.set_xlabel("Real gradient norm", fontsize=9)
ax5.set_title(
    "Step 5: Backpropagation\nactual per-block gradient norm",
    fontsize=9,
    fontweight="bold",
)
ax5.invert_yaxis()
ax5.legend(fontsize=7, loc="lower right")

# Step 6: Real weight update -- shown at enough precision to actually see the nudge
ax6 = axes[1, 2]
ax6.text(
    0.5,
    0.88,
    "Step 6: Update Weights",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.58,
    f"W_old = {old_weight:.6f}\ngradient = {sample_grad:.3e}\nLR = {lr:.0e}",
    ha="center",
    va="center",
    fontsize=10,
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.28,
    f"\u0394W = {weight_delta:+.3e}\nW_new = {new_weight:.6f}",
    ha="center",
    va="center",
    fontsize=11,
    transform=ax6.transAxes,
    color="green",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="lightgreen", alpha=0.5),
)
ax6.axis("off")
ax6.legend(
    handles=[
        Patch(
            facecolor="lightgreen",
            alpha=0.5,
            edgecolor="black",
            label="New (updated) weight",
        )
    ],
    loc="lower center",
    fontsize=7,
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"\n{'=' * 80}")
print(
    "Training Step Summary (all numbers above are real, from one forward+backward pass):"
)
print(f"{'=' * 80}")
print(f"1. Tokenize: {real_len} real tokens + {128 - real_len} padding tokens")
print("2. Mask layout: labels = input shifted left; padding positions set to -100")
print(f"3. Forward pass: logits shape {tuple(outputs.logits.shape)}")
print(f"4. Loss (avg over real tokens only): {step_loss.item():.3f}")
print(f"5. Backprop: gradients computed for all {n_blocks} transformer blocks")
print(
    f"6. Update: W_new = W_old + \u0394W, where \u0394W = -lr * gradient = {weight_delta:+.3e}"
)
print(
    "   That's why fine-tuning needs many steps: each one nudges a weight by a fraction of a "
    "percent, and Riverside's assistant only 'learns' after thousands of these tiny nudges add up."
)
print(f"{'=' * 80}")

---

## The Fine-Tuning Journey: A Problem-Solution Narrative

Rather than a flat taxonomy, fine-tuning is best understood as a **journey where each technique
solves a problem left by the previous one**:

```mermaid
flowchart TD
    A[Pretrained Base Model] -->|no domain knowledge| B[Continued Pretraining]
    B -->|can't follow instructions| C[Instruction Tuning SFT]
    C -->|outputs misaligned with preferences| D[Preference Alignment DPO/RLHF]

    style A fill:#e1f5ff,color:#01579b,stroke:#01579b,stroke-width:1px
    style B fill:#b3e5fc,color:#01579b,stroke:#01579b,stroke-width:1px
    style C fill:#81d4fa,color:#013a63,stroke:#013a63,stroke-width:1px
    style D fill:#4fc3f7,color:#012a4a,stroke:#012a4a,stroke-width:1px
```

(Node fill and text color are both pinned explicitly above -- rather than left to the theme -- so the
diagram stays readable whether your Jupyter/VS Code viewer is in light or dark mode.)

At each stage, you can choose **how many parameters to update**:

| Approach                             | Trade-off                    | Use when                       |
| ------------------------------------ | ---------------------------- | ------------------------------ |
| **Full fine-tuning** (100% params)   | Max quality, max cost        | Small models, abundant compute |
| **Partial freezing** (10-30% params) | Middle ground                | Limited compute budget         |
| **LoRA** (well under 1% params)      | Min cost, swappable adapters | Most production scenarios      |

**This notebook demonstrates:**

- All 3 data-based stages (continued pretraining, instruction tuning, preference alignment)
- All 3 parameter-based approaches (full, partial, LoRA)
- **Explained but not implemented in code:** PPO-based RLHF's optimization mechanics (clipped
  surrogate objective + KL penalty) are contrasted conceptually with DPO's approach in the DPO vs.
  PPO comparison further down (Concept 3) -- not a bare mention, but there's no runnable PPO training
  loop in this notebook.
- **Named but out of scope:** adapters, prefix tuning, QLoRA


## Concept 1 (Data-Based): Non-Instructional Fine-Tuning (Continued Pretraining)

**Riverside's question for this section:** does the model even know our characters and world exist
yet? Nothing downstream matters if it can't recognize "Aria Voss" or "the Meridian's Promise."

**What it is:** keep training with the exact same objective used for the original pretraining --
next-token prediction -- but on your own raw, unlabeled domain text instead of general web text. No
prompts, no "instructions", no labeled pairs: just plain paragraphs. This is often called _continued
pretraining_ or _domain-adaptive pretraining (DAPT)_.

**When to use it:** you have a pile of domain text (support tickets, legal filings, a publisher's back
catalog...) and you want the model to _absorb_ its vocabulary, facts, and style before you ever teach
it to follow instructions.

**Pros**

- Cheapest data to acquire -- no labeling/annotation needed, just clean text.
- Great at absorbing vocabulary, entities, and stylistic quirks (character names, invented
  terminology...).
- Simple training loop -- identical to pretraining (`labels = input_ids`).

**Cons**

- Does **not** teach the model to follow instructions or hold a conversation -- it only gets better
  at _continuing_ text like your domain text.
- Risk of shallow memorization instead of generalization if the corpus is small or repetitive.
- Risk of **catastrophic forgetting** of general-purpose ability if trained too long/aggressively.

Below we run this on a sample of chapters from Riverside's catalog, updating **all** of
`gpt2-medium`'s parameters (full fine-tuning -- more on that axis further down).


### Code Walkthrough: `tokenize_causal()` — Preparing Text for Next-Token Prediction

This is the first point in the notebook where we actually need to convert raw paragraph strings into
the fixed-length integer tensors a transformer consumes, so this is where `tokenize_causal()` gets
defined, right before the `dataset.map(...)` call that needs it. Every later stage that trains on
plain continuation text (partial freezing and LoRA continued pretraining, further down) reuses this
exact same function; the instruction-tuning and DPO sections swap in a response-masked variant
instead, since those need to hide the prompt from the loss.

```python
def tokenize_causal(examples, tokenizer, max_length=128):
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens
```

**Arguments:**

| Argument     | Type                  | Purpose                                                                                                                                                                                                                                        |
| ------------ | --------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `examples`   | `dict` (HF batch)     | A batch of examples with a `"text"` column — here, the raw paragraph strings in `non_inst_paragraphs`, wrapped in a `Dataset`.                                                                                                                 |
| `tokenizer`  | `PreTrainedTokenizer` | The tokenizer loaded in the baseline cell above (`AutoTokenizer.from_pretrained(MODEL_NAME)`). Passed in explicitly rather than closed over, so the same function works unchanged no matter which model/tokenizer this notebook is pointed at. |
| `max_length` | `int`, default `128`  | Hard cap on sequence length. Longer paragraphs are truncated; shorter ones are padded up to this length so every example in a batch has the same shape.                                                                                        |

**What it returns:** the usual tokenizer output (`input_ids`, `attention_mask`) plus a `labels` key,
which HuggingFace's `Trainer` requires to compute the causal-LM loss. `labels` starts as a copy of
`input_ids`, then every padding position (where `attention_mask == 0`) is overwritten with `-100` —
PyTorch's `CrossEntropyLoss` convention for "ignore this position." Without that mask, the model would
waste training signal learning to predict padding tokens instead of real text.

It's called in the cell below as `dataset.map(lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"])`.
`batched=True` is what makes `examples` a dict-of-lists (every text in the batch at once) instead of a
single example, which is why the list comprehension inside zips over `tokens["input_ids"]` rather than
indexing a single sequence.


### What Happens When a Paragraph Is Longer Than `max_length`?

The table above says "longer paragraphs are truncated" as if that's the only option available --
it's actually one of two industry-standard strategies, and `tokenize_causal()` only implements the
first one.

**What `truncation=True` actually does:** it's a paper cutter, not a compressor. Configure
`max_length=512` and hand it an 800-token document, and it keeps the first 512 tokens and
**permanently deletes** the remaining 288 -- no gradient is ever computed for the deleted tail. That's
exactly what happens to any Riverside paragraph past 128 tokens in the cell below.

**The two real strategies, side by side:**

| Training type                           | Strategy                               | What happens to long text                                                                                                                                                                              |
| --------------------------------------- | -------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **Instruction tuning (SFT)**            | **Strict truncation**                  | Cut off at `max_length`; the trailing text is discarded so each `(prompt, completion)` pair keeps its structure intact. This is what `tokenize_causal()` does.                                         |
| **Pretraining / continued pretraining** | **Packing (concatenation & chunking)** | Every document in the corpus is concatenated into one continuous token stream (documents separated by an EOS/separator token), then sliced into contiguous, fixed-length blocks. No text is discarded. |

Packing doesn't lose text, but it does **fragment** it: a 1,500-token article sliced into 1,024-token
blocks lands with its first 1,024 tokens in Block A and its remaining 476 tokens at the _start_ of
Block B -- split wherever the block boundary happens to fall, not at a sentence or paragraph edge.

### Do We Overlap Blocks the Way RAG Chunking Does?

No. Standard causal-LM training loops use a stride exactly equal to the block size -- zero overlap
by default.

**Why not:** compute cost. RAG can afford to re-embed overlapping chunks because inference-time
compute is cheap, and missing a fact split across a chunk boundary is the worse failure mode.
Pretraining makes the opposite trade: a 50% overlap (stride 512 on a 1,024-token block) means running
the full self-attention forward _and_ backward pass over the same tokens twice -- and pretraining runs
across trillions of tokens, so that roughly doubles an already multi-million-dollar compute bill. The
traditional industry default has been: skip the overlap, spend that compute budget on more unique data
instead.

### The Real Cost of No Overlap: Boundary Fragmentation

Skipping overlap isn't free, though -- it creates a structural blind spot at the edge of every block.
Causal attention only looks left, so consider that same 1,500-token article split across two blocks:

```text
BLOCK A: [ Token 0, Token 1, ... Token 1023 ]
         ^ 1,023 tokens of real left-context to predict Token 1023. Plenty of signal.

BLOCK B: [ Token 1024, Token 1025, ... ]
         ^ Attention resets here -- Token 1024 is now "index 0" of a brand-new sequence.
           It has ZERO real left-context, even though it's the direct continuation of Block A.
```

The loss still penalizes the model for failing to predict Token 1025 well -- even though the context
that would have made that easy (everything in Block A) sits in a completely separate training example
the model can't see. Every block boundary in a packed dataset repeats this: tokens near the start of
each block get trained on artificially weak context, not because the information doesn't exist, but
because packing severed the seam between it and the token trying to use it.

### How Modern Data Engineering Actually Reduces This

Because of exactly this problem, most production pipelines don't rely on plain "concatenate and chop":

- **Block-diagonal masking (pack isolation):** pack several _short_ documents into one block, but
  customize the attention mask so tokens can't attend across document boundaries -- solves
  cross-document leakage, though it doesn't help documents that must themselves be split.
- **Best-fit bin packing:** sort documents by length and group them like Tetris pieces so as many as
  possible fit whole inside one block -- fewer documents need splitting at all.
- **Seamless / sliding packing:** newer techniques that introduce a small, deliberately managed
  overlap only at the exact point a long document has to be split -- healing continuity locally
  instead of doubling compute across the entire corpus.


> **PyTorch → Keras:** `from datasets import Dataset` / `from transformers import Trainer, TrainingArguments` / `dataset.map(...)` — HuggingFace's `Dataset.map()` applies `tokenize_causal()` to every example (batched, for speed), producing the `input_ids`/`attention_mask`/`labels` columns that `Trainer` (a full PyTorch training-loop wrapper: batching, forward/backward, optimizer step) consumes next. **Keras/TF equivalent:** `tf.data.Dataset.map(...)` + `model.fit(...)` — a Keras version would build a `tf.data.Dataset` pipeline with the same `.map()` call and then call the standard `model.fit(dataset, epochs=...)` in place of HuggingFace's `Trainer` (or use `TFAutoModelForCausalLM` with HuggingFace's own `Trainer`, which wraps `model.fit` under the hood).

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

non_inst_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery"],
    max_chapters=None,  # None = all available chapters per novel
)
print(
    f"Loaded {len(non_inst_paragraphs)} paragraphs from 3 novels for continued-pretraining demo"
)

non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})  # wrap the paragraph list in a HF Dataset


def tokenize_causal(examples, tokenizer, max_length=128):
    """Standard next-token-prediction tokenization: labels = input_ids, with padding
    positions masked out (-100) so the loss ignores them."""
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )

    # Copy input ids into labels, replacing padding positions with -100 so the loss skips them
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


non_inst_tokenized = non_inst_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)  # apply tokenize_causal across the whole dataset in batches, dropping the raw text column


Data's ready. Now load a **fresh, untouched copy** of `gpt2-medium` to actually fine-tune -- kept
separate from `base_model` so `base_model` stays the permanent "before" snapshot every later
comparison in this notebook relies on.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` / `p.numel()` — loads a second, independent copy of `gpt2-medium` (kept separate from `base_model` so the original stays an untouched "before" snapshot) and `.numel()` counts the total scalar elements in each parameter tensor to report the full trainable-parameter count. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` / `model.count_params()` — Keras models expose a built-in `count_params()` method that sums all trainable + non-trainable weight sizes in one call, instead of manually summing `p.numel()` over every parameter.

In [ ]:
full_ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)  # separate fresh copy dedicated to full fine-tuning
print(
    f"Loaded a fresh {MODEL_NAME} to fully fine-tune: "
    f"{sum(p.numel() for p in full_ft_model.parameters()):,} parameters, all trainable."
)


### Configuring and Running the Trainer

`TrainingArguments` + `Trainer` is HuggingFace's standard training loop -- it handles the batching,
the forward/backward pass, and the optimizer step described earlier in this notebook, so we don't
write that loop by hand. `max_steps=60` and `learning_rate=5e-5` keep this CPU demo fast; a real
Riverside training run would raise `max_steps` substantially. `trainer_full.train()` is the line that
actually runs all 60 of those steps -- this is the real training run every later comparison in this
notebook is measured against.


> **PyTorch → Keras:** `TrainingArguments(...)` / `Trainer(model=..., args=..., train_dataset=...)` / `trainer_full.train()` / `full_ft_model.save_pretrained(...)` — configures and runs HuggingFace's full PyTorch training loop (batching, forward/backward passes, optimizer steps, logging) in one `.train()` call, then serializes the fine-tuned weights + config to disk. **Keras/TF equivalent:** `model.compile(optimizer=..., loss=...)` + `model.fit(dataset, epochs=...)` — the direct Keras analog of configuring + running training; `save_pretrained(...)` has an identically-named method on `TFPreTrainedModel` subclasses, so the checkpoint-saving line itself would be unchanged in a TF version.

In [ ]:
# Configure a short training run (max_steps kept small for CPU-friendly demo purposes)
training_args_full = TrainingArguments(
    output_dir="./checkpoints/non-instruction-full",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle; raise further for real runs
    logging_steps=10,
    save_strategy="no",
    learning_rate=5e-5,
    report_to="none",
)

# Wrap the model, config, and tokenized dataset in a Trainer and run the actual training loop
trainer_full = Trainer(
    model=full_ft_model, args=training_args_full, train_dataset=non_inst_tokenized
)
trainer_full.train()
full_ft_model.save_pretrained("./checkpoints/non-instruction-full")  # persist weights to disk for later reload
print("Saved continued-pretraining (full fine-tune) checkpoint.")


### Weight Movement Layer by Layer

The bar chart above shows each block's _total_ movement collapsed into one number - the L2 norm of all
its deltas combined. That is useful for comparing blocks, but it loses the distribution of movement
_within_ each block and does not show whether large and small changes are clustered or uniformly
spread.

The cell below samples the same number of individual weight deltas from every transformer block, then
gives **each block its own panel**. Figures contain at most 10 panels, so `gpt2-medium`'s 24 blocks are
shown as blocks 0-9, 10-19, and 20-23. Every panel uses the same y-axis scale: a quiet block therefore
cannot look as active as a strongly moving block merely because its axis was automatically rescaled.

> **What to look for:** Compare the mean and maximum in each panel title, then inspect the shape. Long
> regions near zero mean many sampled weights barely moved; spikes identify sampled weights with larger
> updates. A concentration of higher means or taller spikes in later blocks suggests those layers
> adapted more strongly during continued pretraining.

> **PyTorch → Keras:** `base_model.named_parameters()` / `p.data.cpu()` / `torch.cat(block_deltas)` — snapshots every parameter's raw values before comparing against the reloaded fine-tuned checkpoint; `.cpu()` ensures both tensors being subtracted live on the same device, and `torch.cat` concatenates a block's many per-tensor deltas into one flat array for sampling/plotting. **Keras/TF equivalent:** `model.get_weights()` / `tf.concat(...)` — Keras's `get_weights()` returns a plain list of NumPy arrays (no device handling needed since the conversion already happened), and `tf.concat` is the direct equivalent of `torch.cat` for combining several weight-delta arrays into one.

In [ ]:
# Load the saved checkpoint (full_ft_model was freed above; reload from disk for the layer panels)
# Snapshot the pre-fine-tune weights (base_model is the permanent "before" snapshot), so
# we can diff against them below.
import gc

base_state = dict(base_model.named_parameters())
ft_trace_model = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to("cpu")

SAMPLES_PER_BLOCK = 500  # weights sampled per transformer block
PANELS_PER_FIGURE = 10
N_COLUMNS = 2

all_deltas = []
for block_i in range(base_model.config.n_layer):  # walk every transformer block in order
    block_deltas = []
    for name, parameter in ft_trace_model.named_parameters():
        if f"h.{block_i}." in name:  # only this block's own parameters
            delta = (
                parameter.data.cpu() - base_state[name].data.cpu()
            ).abs().flatten()  # absolute weight movement vs. the untouched base checkpoint
            block_deltas.append(delta)
    if block_deltas:
        combined = torch.cat(block_deltas)
        stride = max(1, len(combined) // SAMPLES_PER_BLOCK)  # even subsampling so every block plots the same count
        sampled = combined[::stride][:SAMPLES_PER_BLOCK].numpy()
        all_deltas.append(sampled)

# Use one shared scale across every figure. Without this, a quiet block could look as active as
# the block with the largest movement simply because Matplotlib rescaled its panel.
global_ymax = max(float(block.max()) for block in all_deltas)
y_limit = global_ymax * 1.05 if global_ymax > 0 else 1e-9

for page_start in range(0, len(all_deltas), PANELS_PER_FIGURE):  # paginate blocks across multiple figures
    page = all_deltas[page_start : page_start + PANELS_PER_FIGURE]
    n_rows = (len(page) + N_COLUMNS - 1) // N_COLUMNS

    fig, axes = plt.subplots(
        n_rows,
        N_COLUMNS,
        figsize=(14, 2.6 * n_rows),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()

    for panel_i, block_arr in enumerate(page):  # one subplot per transformer block on this page
        block_i = page_start + panel_i
        weight_indices = np.arange(len(block_arr))
        axis = axes[panel_i]

        axis.plot(weight_indices, block_arr, linewidth=0.7, color="steelblue")
        axis.fill_between(weight_indices, block_arr, alpha=0.12, color="steelblue")
        axis.set_title(
            f"Transformer block {block_i}  "
            f"(mean={block_arr.mean():.2e}, max={block_arr.max():.2e})",
            fontsize=9,
        )
        axis.set_ylim(0, y_limit)
        axis.grid(alpha=0.2, axis="y")

    for unused_axis in axes[len(page) :]:
        unused_axis.set_visible(False)  # hide any empty grid cells on the last page

    page_end = page_start + len(page) - 1
    fig.suptitle(
        f"Full Fine-Tuning Weight Movement: Blocks {page_start}-{page_end}",
        fontsize=12,
        fontweight="bold",
    )
    fig.supxlabel(f"Sampled weight index ({SAMPLES_PER_BLOCK} weights per block)")
    fig.supylabel("|W_after - W_before|")
    plt.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.show()

peak_block = max(range(len(all_deltas)), key=lambda i: all_deltas[i].mean())  # block with the largest average movement
min_block = min(range(len(all_deltas)), key=lambda i: all_deltas[i].mean())  # block with the smallest average movement
print(
    f"Mean |delta W| by block - min: block {min_block} "
    f"({all_deltas[min_block].mean():.4e}),  "
    f"max: block {peak_block} ({all_deltas[peak_block].mean():.4e}).  "
    f"Each panel shows {SAMPLES_PER_BLOCK} sampled individual-weight deltas."
)

del ft_trace_model
gc.collect()  # free the reloaded model's memory now that its deltas are captured
print("Freed ft_trace_model from memory (checkpoint still on disk).")


### Visualizing Training Progress: Loss Curves

After training completes, let's look at what actually happened -- the real per-step loss recorded by
the `Trainer` above, not an idealized illustration. Textbook loss curves are smooth; a 60-step,
batch-size-2 CPU demo on a fresh model is usually much noisier, and that's worth seeing honestly.

**What to look for:**

1. **Downward trend:** Loss should decrease on average (model is learning), even if noisy step-to-step
2. **Convergence:** Loss should stop trending strongly downward by the end (not still falling fast)
3. **Magnitude:** Lower loss = better fit to domain text (but watch for overfitting on tiny corpora!)

**Reading a noisy real curve:** with only 6 logged points and batch_size=2, a single unusually easy or
hard paragraph can swing the reported loss by ±0.3 or more. Don't over-interpret small wiggles -- look
at the overall direction across all points, and compare against the other techniques' real curves
later in the notebook (instruction tuning, partial freezing, LoRA continued pretraining) to see which
setup is converging fastest for the same step budget.


> **PyTorch → Keras:** `trainer.state.log_history` — HuggingFace's `Trainer` records a running list of dicts (step number, loss, learning rate, etc.) logged every `logging_steps`; this cell filters that list down to just the `(step, loss)` pairs for plotting. **Keras/TF equivalent:** `history = model.fit(...)` / `history.history["loss"]` — Keras's `fit()` returns a `History` object whose `.history` dict holds per-*epoch* (not per-step, by default) metric lists; matching HuggingFace's per-step granularity in Keras needs a custom callback (e.g. overriding `on_train_batch_end`).

In [ ]:
# Visualize the REAL loss curve from the continued-pretraining run above (trainer_full),
# not a fabricated "typical" curve -- this is exactly what your training just did.
def extract_loss_history(trainer):

    # Pull (step, loss) pairs out of the Trainer's log history, skipping eval-only entries
    return [
        (entry["step"], entry["loss"])
        for entry in trainer.state.log_history
        if "loss" in entry
    ]


full_ft_history = extract_loss_history(trainer_full)
steps, losses = zip(*full_ft_history)  # split into two parallel sequences for plotting

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    steps,
    losses,
    marker="o",
    linewidth=2,
    markersize=7,
    color="green",
    label="Training loss",
)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title(
    f"Continued Pretraining (Full FT): real loss log ({losses[0]:.2f} \u2192 {losses[-1]:.2f})",
    fontsize=12,
    fontweight="bold",
)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\n{'=' * 70}")
print("Reading this REAL loss curve (not an idealized one):")
print(f"{'=' * 70}")
print(f"  Logged steps: {list(steps)}")
print(f"  Logged losses: {[round(l, 3) for l in losses]}")
print(f"  First -> last: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"{'=' * 70}")
print("What to look for:")
print(
    "  • A clean, monotonic plateau like a textbook figure is the exception, not the rule --"
)
print(
    "    especially at batch_size=2 with only a handful of steps, loss is dominated by"
)
print("    per-batch noise (which paragraph happened to be in this batch) more than by")
print("    the underlying trend.")
print(
    "  • If the trend is flat/noisy rather than decreasing: raise max_steps, increase the"
)
print(
    "    batch size, or train on more paragraphs so the trend has room to dominate the noise."
)
print(
    "  • Compare this to the loss curves for instruction tuning, partial freezing, and LoRA"
)
print(
    "    continued pretraining further down -- they were all recorded the same real way."
)
print(f"{'=' * 70}")


### Common Pitfalls: Continued Pretraining

**Pitfall #1: Catastrophic Forgetting**

**Bad:** Train for 10,000 steps on a tiny 50KB domain corpus  
**Good:** Train for 25-100 steps, then validate on general tasks (e.g., "The capital of France is...")

**Why it happens:** The model "overwrites" its general language knowledge with domain-specific patterns.

**How to avoid:**

- Keep training steps low initially (start with 25-50)
- Use a validation set with both domain AND general questions
- Watch for nonsense on general prompts (sign of forgetting)

---

**Pitfall #2: Shallow Memorization**

**Bad:** Tiny corpus (5KB), repeated 100 times → model memorizes exact phrases  
**Good:** Diverse corpus (500KB+) with varied writing styles

**How to detect:**

- Model completes prompts with **exact** training sentences (word-for-word)
- Model can't generalize to new prompts in the same style
- Perplexity drops close to its theoretical floor of **1.0** on training data (the model is nearly
  certain about every next token because it has seen this exact text before) but stays high on
  validation data it hasn't memorized

---

**Pitfall #3: Wrong Max Length**

**Bad:** `max_length=512` on a corpus of short sentences → 90% of every batch is padding  
**Good:** Match `max_length` to your typical paragraph length (128-256 for novels)

**Why it matters:** Wasted computation on padding, slower training, less effective learning

---

**Pitfall #4: No Tokenizer Padding Token**

**Bad:** Forget to set `tokenizer.pad_token` → crash or silent errors  
**Good:** Always set `tokenizer.pad_token = tokenizer.eos_token` for GPT-family models

---

**Quick Health Check After Training:**

```python
# Test 1: Domain knowledge (should work)
generate(model, "Aria Voss checked the Meridian's Promise and")

# Test 2: General knowledge (should still work!)
generate(model, "The capital of France is")

# Test 3: Novel generalization (should work, not memorize)
generate(model, "In the Under-Hold, the rebels gathered and")
```

If test 2 fails → you overtrained (catastrophic forgetting).  
If test 3 is word-for-word from training → shallow memorization.


## Concept 2 (Data-Based): Instructional (Supervised) Fine-Tuning

**Riverside's question for this section:** the model now knows the lore -- but can a ghostwriter
_ask_ it for a continuation, or does it just ramble? An assistant nobody can direct isn't an
assistant.

### The Problem with Continued Pretraining Alone

After continued pretraining, the model knows your domain vocabulary and can continue text in your
style. But try asking it a question:

**You:** `"What are the five tides in the Tidebound world?"`  
**Model (after continued pretraining):** `"What are the five tides in the Tidebound world? This 
question has puzzled scholars for centuries. Some believe there are actually six tides, while..."`
(continues rambling)

**The problem:** The model learned to _continue_ prose, not to _answer questions_ or _follow
instructions_. It will keep generating narrative-style text forever because that's what it was trained
on.

### The Solution: Instruction Tuning (Supervised Fine-Tuning / SFT)

**What it is:** Train on `(prompt, completion)` pairs where the **prompt** is an instruction/question
and the **completion** is the desired response. Crucially, we **mask the prompt tokens** in the loss
so the model is only penalized for the completion portion.

**Key insight:** This teaches the model _behavior_ -- "when you see input shaped like X, respond like
Y" -- rather than just "keep talking like this corpus."

Real-world instruction datasets include:

- [alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned) - 52K instruction-following examples
- [OpenOrca](https://huggingface.co/datasets/Open-Orca/OpenOrca) - 4.2M GPT-4 completions
- [OpenAssistant/oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1) - 161K human-rated conversations

Here we auto-derive a tiny instruction dataset from Riverside's catalog:

- **Prompt:** `"Continue the fiction narrative in the same style: <paragraph N>"`
- **Completion:** `<paragraph N+1>`

**Pros:**

- Model learns to _follow a format/instruction_, not just continue prose
- Directly usable for chat/assistant interfaces
- Loss masking means the model isn't penalized for "predicting" the prompt it didn't generate

**Cons:**

- Needs actual (prompt, completion) pairs (expensive to create by hand)
- Can narrow diversity toward the exact template it was trained on
- Doesn't fix preference issues (model might follow instructions but in an unhelpful way)

This cell introduces **LoRA** (parameter-efficient tuning) to keep training fast on CPU -- foreshadowing
the budget conversation Riverside's IT lead is going to have with us later.


> **PyTorch → Keras:** `from peft import LoraConfig, get_peft_model, TaskType` — imports HuggingFace's PEFT library, which wraps a PyTorch model's targeted `nn.Linear` layers with low-rank adapter matrices and freezes everything else; the actual wrapping happens a few cells down. **Keras/TF equivalent:** there is no first-party `peft` support for `TFPreTrainedModel`s — the common Keras/TF pattern for parameter-efficient tuning is manual layer freezing (`layer.trainable = False` on all but the last few layers) rather than LoRA adapters, since PEFT's LoRA implementation is PyTorch-only.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

INSTRUCTION_PREFIX = "Continue the fiction narrative in the same style:\n\n"


def build_instruction_pairs(novels=None, max_chapters=None):
    if novels is None:
        novels = [
            "scifi",
            "fantasy",
            "mystery",
            "cyberpunk",
            "literary",
        ]  # 5-genre default for richer style diversity

    pairs = []
    for alias in novels:  # walk every requested novel
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        for path in sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]:  # each chapter file, capped if requested
            paras = [
                p.strip().replace("\n", " ")
                for p in path.read_text(encoding="utf-8").split("\n\n")
                if len(p.strip()) > 200
            ]  # split the chapter into paragraphs, dropping ones too short to be useful
            for a, b in zip(paras, paras[1:]):  # pair each paragraph with the one right after it
                pairs.append(
                    {"prompt": f"{INSTRUCTION_PREFIX}{a}\n\n", "completion": b}
                )
    return pairs


### Generating Instruction Pairs at Scale: Tools and Models

This notebook derives instruction pairs from consecutive paragraph pairs — a zero-dependency approach
that works directly on Riverside's proprietary corpus without sending any text to an external API.
In production settings with larger corpora, several purpose-built tools and models exist for
auto-generating higher-quality instruction pairs from raw text:

| Approach                         | Tool / Model                                                                  | How it works                                                                        | Best for                                                     |
| -------------------------------- | ----------------------------------------------------------------------------- | ----------------------------------------------------------------------------------- | ------------------------------------------------------------ |
| **Self-Instruct**                | [self-instruct](https://github.com/yizhongw/self-instruct) (Wang et al. 2022) | Uses an LLM to bootstrap instruction-following data by prompting it with a seed set | Diverse instruction coverage from any raw corpus             |
| **Alpaca pipeline**              | [Stanford Alpaca](https://github.com/tatsu-lab/stanford_alpaca)               | Used `gpt-3.5-turbo` to generate 52K instruction pairs from 175 seed examples       | One-time generation of a large, clean SFT dataset            |
| **Evol-Instruct**                | [WizardLM](https://github.com/nlpxucan/WizardLM)                              | Starts with simple instructions and iteratively "evolves" them into harder variants | Improving model reasoning and instruction diversity          |
| **Open-source LLM as generator** | Llama-3.1-8B-Instruct, Mistral-7B-Instruct                                    | Zero-shot: `"Convert this paragraph into an instruction-response pair: <text>"`     | On-device, fully private generation — ideal for Riverside    |
| **GPT-4o structured output**     | `openai` library, `response_format={"type": "json_object"}`                   | Generates clean `{"instruction": ..., "response": ...}` JSON in one API call        | Highest quality pairs when external API access is acceptable |

**Libraries and datasets worth knowing:**

- [`alpaca_farm`](https://github.com/tatsu-lab/alpaca_farm): full pipeline (data gen, SFT, RLHF, eval) based on the Stanford Alpaca approach
- [`LLM2SFT`](https://github.com/shibing624/LLM2SFT): lightweight pipeline for converting arbitrary documents to SFT format
- [Hugging Face `datasets`](https://huggingface.co/datasets?task_categories=task_categories:text-generation&sort=trending): search for `instruction` — several pre-built datasets (alpaca-cleaned, OpenOrca, Dolly 15K) are ready to download

**For Riverside's use case** (private manuscripts, no data leaving the building), the most practical option would be running a quantized Mistral-7B-Instruct locally and prompting it to convert each paragraph into an instruction-response pair — the same `"Continue the fiction narrative:"` framing used here, just generated by a separate model rather than extracted by simple pairing.


### Tokenizing With the Prompt-Mask Pattern

`build_instruction_pairs()` gave us `(prompt, completion)` strings; `tokenize_instruction()` converts
one pair into the actual tensors `Trainer` needs, using the same prompt-masking idea introduced
earlier in this notebook: `labels` starts as a full copy of the tokenized text, then every **prompt**
position (not just padding) gets set to `-100`, so the loss only ever grades the completion.


> **PyTorch → Keras:** `tokenize_instruction()` — builds `labels` as a copy of the tokenized `input_ids`, then overwrites *both* the prompt-token positions and the padding positions with `-100`, so a cross-entropy loss with `ignore_index=-100` (used earlier in the notebook) only ever grades the completion tokens. **Keras/TF equivalent:** the same masking logic — a Keras/TF version would build an analogous `labels` array with `-100` (or `0` plus a matching `sample_weight` mask, since TF's `SparseCategoricalCrossentropy` has no built-in `ignore_index`) at prompt+padding positions; the tokenization itself is identical since `AutoTokenizer` is framework-agnostic.

In [ ]:
def tokenize_instruction(example, max_length=160, prompt_max_length=96):
    prompt_ids = tokenizer(
        example["prompt"], truncation=True, max_length=prompt_max_length
    )["input_ids"]  # tokenize just the prompt so we know where it ends
    full_text = example["prompt"] + example["completion"]
    tokens = tokenizer(
        full_text, truncation=True, padding="max_length", max_length=max_length
    )  # tokenize the full prompt+completion sequence together
    labels = tokens["input_ids"].copy()
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100  # don't compute loss on the prompt portion
    for i, mask in enumerate(tokens["attention_mask"]):
        if mask == 0:
            labels[i] = -100  # don't compute loss on padding either
    tokens["labels"] = labels
    return tokens


### Building the Instruction Dataset

Now actually build the pairs from 5 genres and tokenize every one of them with the function above.


In [ ]:
instruction_pairs = build_instruction_pairs(
    novels=["scifi", "fantasy", "mystery", "cyberpunk", "literary"],
    max_chapters=None,  # None = all available chapters
)
print(f"Built {len(instruction_pairs)} instruction pairs from 5 novels")

instruction_dataset = Dataset.from_list(instruction_pairs)  # wrap the prompt/completion pairs in a HF Dataset
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction, remove_columns=["prompt", "completion"]
)  # apply the prompt-masking tokenizer across every pair


### Wrapping the Base Model in a LoRA Adapter

This is the first LoRA usage in the notebook -- see the dedicated LoRA section further down for the
full math; for now, `get_peft_model()` freezes every weight in a fresh base model and injects small
trainable adapter matrices into each `c_attn` projection, so only those tiny matrices get optimizer
state during training.


> **PyTorch → Keras:** `LoraConfig(...)` / `get_peft_model(instruct_base, lora_config)` / `.print_trainable_parameters()` — `get_peft_model()` freezes every weight in the base PyTorch model, then injects small trainable low-rank adapter matrices into each layer named in `target_modules` (here, GPT-2's combined `c_attn` projection); `.print_trainable_parameters()` reports what fraction of the total is now trainable. **Keras/TF equivalent:** no direct equivalent — since PEFT's LoRA wrapping is PyTorch-only, a Keras/TF version would instead freeze whole layers manually (`for layer in model.layers[:-k]: layer.trainable = False`), a coarser, layer-level approximation of LoRA's finer-grained low-rank adapters.

In [ ]:
# LoRA hyperparameters: rank-8 adapters injected into GPT-2's attention projection only
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT-2's combined attention projection
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)  # fresh base copy dedicated to this LoRA run
instruct_lora_model = get_peft_model(instruct_base, lora_config)  # freeze base weights, inject trainable LoRA adapters
instruct_lora_model.print_trainable_parameters()


### Training and Saving the Adapter

Same `Trainer` pattern as continued pretraining, just with the LoRA-wrapped model, the prompt-masked
dataset, and a higher learning rate (`2e-4` vs. `5e-5`) -- LoRA needs a higher LR since it's only
updating a tiny slice of parameters.


> **PyTorch → Keras:** `Trainer(model=instruct_lora_model, ...)` / `trainer_instruct.train()` / `instruct_lora_model.save_pretrained(...)` — the same HuggingFace `Trainer` pattern as the earlier full fine-tuning run, just pointed at the LoRA-wrapped model and the prompt-masked instruction dataset, with a higher learning rate since only the small adapter matrices are being updated. **Keras/TF equivalent:** `model.fit(dataset, epochs=...)` — as with the earlier full fine-tuning cell, a Keras/TF version would call `.fit()` on the (layer-frozen) model instead of `Trainer.train()`; `save_pretrained()` again has an identically-named counterpart on `TFPreTrainedModel`.

In [ ]:
# Configure a short LoRA training run with a higher LR (only the adapter matrices are trainable)
training_args_instruct = TrainingArguments(
    output_dir="./checkpoints/instruction-lora",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

# Wrap the LoRA-wrapped model, config, and tokenized instruction dataset in a Trainer and run it
trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained("./checkpoints/instruction-lora")  # persist the adapter weights only
print("Saved instruction-tuned LoRA adapter.")


### Code Walkthrough: Instruction Tuning, Recapped

**What just ran, across the last several cells -- four conceptual steps, each already introduced
briefly right before its own code. This cell ties them together with the details that don't fit in
a one-paragraph intro:**

---

**Step A: `build_instruction_pairs()` — creating (prompt, completion) pairs**

```python
pairs.append({"prompt": f"{INSTRUCTION_PREFIX}{a}\n\n", "completion": b})
```

For every pair of consecutive paragraphs `(a, b)` in each chapter, the _preceding_ paragraph becomes the prompt and the _next_ paragraph becomes the completion. This is the cheapest way to auto-generate instruction pairs from raw prose — no human labelling required. The `"\n\n"` delimiter marks where the model should stop echoing the prompt and start generating.

---

**Step B: `tokenize_instruction()` — the prompt-mask pattern**

This is the core difference from continued pretraining's `tokenize_causal()`:

```
Continued pretraining:  [-100 for padding only,  real labels for everything else]
Instruction tuning:     [-100 for prompt + pad,  real labels for completion only]
```

In code:

```python
for i in range(min(len(prompt_ids), len(labels))):
    labels[i] = -100   # mask the entire prompt portion
```

Setting `labels[i] = -100` at prompt positions tells `F.cross_entropy(ignore_index=-100)` to skip those positions when computing the loss. The model is only graded on the _completion_ tokens — never penalised for "predicting" the instruction it was given.

---

**Step C: `LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"])` — LoRA hyperparameters**

| Param                       | Value          | What it controls                                             |
| --------------------------- | -------------- | ------------------------------------------------------------ |
| `r=8`                       | Rank           | Bottleneck dimension — 8 basis vectors to express the update |
| `lora_alpha=16`             | Scaling        | Effective LR multiplier = `alpha/r` = 2.0                    |
| `target_modules=["c_attn"]` | Which layers   | GPT-2 packs Q, K, V into one `c_attn` projection             |
| `lora_dropout=0.05`         | Regularization | Randomly zeros adapter activations during training           |

`get_peft_model(base, config)` wraps every targeted layer with a `LoraLayer` object and freezes all other weights — the only parameters that get optimizer state are the two small adapter matrices per layer.

---

**Step D: `Trainer.train()` — what HuggingFace's Trainer does for you**

Under the hood, one `Trainer.train()` call:

1. Iterates over `train_dataset` in mini-batches of `per_device_train_batch_size=2`
2. Calls `model.forward(input_ids, attention_mask, labels)` → computes cross-entropy loss (ignoring `labels=-100` positions)
3. Calls `loss.backward()` → computes gradients only for `requires_grad=True` parameters (the LoRA matrices)
4. Calls `optimizer.step()` → updates those parameters by `lr × gradient`
5. Logs the loss every `logging_steps=10` steps

6. Stops after `max_steps=60` regardless of dataset size


### The Instruction-Tuning Mask Layout, For Real

Contrast this with the continued-pretraining mask layout earlier in the notebook: there, only
**padding** was masked, and every real token was active. Here, the **prompt itself is masked too** --
the model is only ever penalized for generating the completion, never for reproducing the prompt it
was given. The cell below takes one real `(prompt, completion)` pair from `instruction_pairs`, runs it
through the real `tokenize_instruction()` used for training, and colors every token position by what
the label mask actually does with it.


In [ ]:
# Real mask layout for instruction tuning: prompt masked (-100), completion active, padding masked
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

example_pair = instruction_pairs[0]
encoded_example = tokenize_instruction(example_pair)
example_labels = np.array(encoded_example["labels"])
example_attention = np.array(encoded_example["attention_mask"])

# Classify every position: 0 = masked prompt, 1 = active completion, 2 = masked padding
region = np.zeros(len(example_labels), dtype=int)
region[example_attention == 0] = 2  # padding
region[(example_attention == 1) & (example_labels != -100)] = 1  # completion (active)

# everything else (attention==1 & labels==-100) is the masked prompt, stays 0

prompt_masked = int(np.sum(region == 0))
completion_active = int(np.sum(region == 1))
padding_masked = int(np.sum(region == 2))

# Visualize the mask layout as a single color-coded strip (gray=prompt, green=completion, white=padding)
fig, ax = plt.subplots(figsize=(14, 2.2))
cmap = ListedColormap(["lightgray", "mediumseagreen", "white"])
ax.imshow(
    region.reshape(1, -1),
    cmap=cmap,
    aspect="auto",
    vmin=0,
    vmax=2,
    extent=[0, len(region), 0, 1],
)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title(
    f"{prompt_masked} prompt tokens masked + {completion_active} completion tokens active "
    f"+ {padding_masked} padding tokens masked",
    fontsize=10,
    fontweight="bold",
)

# Legend entries matching each color band in the strip above
legend_handles = [
    Patch(
        facecolor="lightgray", edgecolor="black", label="Prompt (masked, labels=-100)"
    ),
    Patch(
        facecolor="mediumseagreen",
        edgecolor="black",
        label="Completion (active, real labels)",
    ),
    Patch(facecolor="white", edgecolor="black", label="Padding (masked, labels=-100)"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.55),
    ncol=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(f"Prompt text:     {example_pair['prompt'][:80]!r}...")
print(f"Completion text: {example_pair['completion'][:80]!r}...")
print(
    f"\nMasked (prompt): {prompt_masked} tokens | Active (completion): {completion_active} tokens "
    f"| Masked (padding): {padding_masked} tokens"
)
print(
    "Compare to continued pretraining: there, every real token was active. Here, the prompt is "
    "masked too, so the model only ever gets gradient signal from the completion."
)

### Common Pitfalls: Instruction Tuning

**Pitfall #1: Forgetting to Mask the Prompt**

**Bad:** Compute loss on both prompt AND completion → model "predicts" the prompt it was given  
**Good:** Set prompt tokens to `-100` in labels → only penalized for the completion

**Why it matters:**

Without masking:

```python
labels = [3, 822, 25, ...]  # entire sequence including prompt
```

With masking:

```python
labels = [-100, -100, -100, 822, 25, ...]  # first 10 tokens (prompt) masked
```

The model should only learn to **generate the completion**, not memorize the prompt.

---

**Pitfall #2: Template Over-Fitting**

**Bad:** All training pairs use identical prefix: `"Continue the narrative: ..."`  
**Good:** Vary the instruction format, or accept this if you'll always use that prefix at inference

**What happens:** Model becomes "allergic" to prompts without the exact prefix. If you prompt with
just the raw paragraph (no prefix), it won't know what to do.

**Fix:** Either:

1. Use diverse instruction templates during training
2. Always use the exact same prefix at inference (consistency is key)

---

**Pitfall #3: Completion Too Short/Long**

**Bad:** Completions are 5 tokens on average → model learns to be terse  
**Good:** Completions should match your inference expectation (50-100 tokens for narrative)

**Why:** The model learns the **distribution of lengths** from training. If all completions are short,
it will always generate short responses, even when you want more detail.

---

**Pitfall #4: Wrong Learning Rate**

**Bad:** Use the same LR as pretraining (5e-5) for LoRA  
**Good:** LoRA needs **higher LR** (2e-4 to 5e-4) because you're only updating a tiny subset of params

**Rule of thumb:**

- Full fine-tuning: 5e-5 to 1e-4
- Partial freezing: 1e-4 to 2e-4
- LoRA: 2e-4 to 5e-4

Lower rank → higher LR (more aggressive updates needed).

---

**Quick Health Check After Instruction Tuning:**

```python
# Test 1: With the instruction prefix (should work)
prompt = INSTRUCTION_PREFIX + "Aria checked the Meridian and\\n\\n"
generate(instruct_model, prompt)

# Test 2: Without prefix (will likely fail if over-fitted to template)
generate(instruct_model, "Aria checked the Meridian and")

# Test 3: Novel instruction (should generalize)
prompt = INSTRUCTION_PREFIX + "In the Upper decks, Marcus\\n\\n"
generate(instruct_model, prompt)
```

If test 2 produces nonsense → template over-fitting (model expects the prefix).  
If test 3 produces off-topic output → not enough diverse training data.


In [ ]:
# Quick health check after instruction tuning

print("=== Test 1: With the instruction prefix ===")
prompt_with_prefix = INSTRUCTION_PREFIX + "Aria checked the Meridian and\n\n"
print(f"  Input : {repr(prompt_with_prefix)}")
print(f"  Output: {generate(instruct_lora_model, prompt_with_prefix)}")
print()

print("=== Test 2: Without the prefix (checks for template over-fitting) ===")
raw_prompt = "Aria checked the Meridian and"
print(f"  Input : {repr(raw_prompt)}")
print(f"  Output: {generate(instruct_lora_model, raw_prompt)}")
print()

print("=== Test 3: Novel instruction (generalization check) ===")
prompt_novel = INSTRUCTION_PREFIX + "In the Upper decks, Marcus\n\n"
print(f"  Input : {repr(prompt_novel)}")
print(f"  Output: {generate(instruct_lora_model, prompt_novel)}")

## Concept 3 (Data-Based): Preference Alignment (DPO)

**Riverside's question:** the model follows instructions now -- but does it write the way editors
_actually_ prefer, not just "a" technically valid response?

**The problem:** Instruction tuning teaches the model to _respond_, but not which responses humans
_prefer_. The same `"Who is Aria Voss?"` prompt might return a ten-paragraph technical inventory
when an editor just wants one clean sentence.

**The solution — DPO (Direct Preference Optimization):** show the model response pairs — one editors
would keep (`chosen`), one they wouldn't (`rejected`) — and train it to assign higher probability to
the preferred one, anchored against a frozen reference snapshot so the model can't cheat by
collapsing onto a single response for everything.

**The data contract** (all that needs to come from outside the library):

| Column     | What it is                                       |
| ---------- | ------------------------------------------------ |
| `prompt`   | The shared input both responses answer           |
| `chosen`   | The preferred response (editors would keep this) |
| `rejected` | The dispreferred response (editors wouldn't)     |

**Where the labels come from in practice:** human editors rating two drafts, implicit usage signals
(which outputs editors actually kept), or an LLM-as-judge. This notebook uses a structural stand-in:
real next paragraph = `chosen`, random unrelated paragraph from a different chapter = `rejected`.

**DPO vs PPO:** PPO trains a separate reward model and uses reinforcement learning; DPO folds that
into a single supervised objective. DPO wins when you already have static `(prompt, chosen, rejected)`
pairs; PPO wins when you need a live, dynamic reward signal (e.g. unit-test pass rates).

**Key pitfalls:** weak contrast between chosen/rejected gives no gradient signal; `beta` too high
collapses outputs to one response; always run DPO _after_ SFT, never on a raw base model.


### DPO vs PPO at a Glance

```mermaid
flowchart TD
    subgraph DPO["DPO — static pairs, no reward model needed"]
        d1["labeled pairs\n(prompt, chosen, rejected)"]
        d2["frozen reference\n(policy snapshot)"]
        d3["DPO loss\nmargin_chosen > margin_rejected?"]
        d4["policy weights updated"]
        d1 --> d3
        d2 --> d3
        d3 --> d4
    end

    subgraph PPO["PPO-RLHF — live rollouts, reward model required"]
        p1["preference data\n(human ratings)"]
        p2["reward model\ntrained separately"]
        p3["policy generates\nlive rollouts"]
        p4["reward model\nscores rollouts"]
        p5["PPO optimizer\nclip(ratio) × reward − β·KL"]
        p6["policy weights updated"]
        p1 --> p2 --> p4
        p3 --> p4 --> p5 --> p6
        p6 -.->|next batch| p3
    end
```

**The structural difference in one sentence:** DPO is a single supervised pass over pre-collected
preference pairs; PPO is a continuous loop where the policy generates its own training data and a
separate reward model grades it in real time.

|                       | DPO                                       | PPO-RLHF                                        |
| --------------------- | ----------------------------------------- | ----------------------------------------------- |
| Needs a reward model? | No                                        | Yes — separate training run                     |
| Data mode             | Static `(prompt, chosen, rejected)` pairs | On-policy live rollouts                         |
| Feedback loop         | None — pairs are fixed                    | Yes — policy re-generates each round            |
| Extra complexity      | Just a frozen reference snapshot          | Reward model + value network + rollout sampling |
| Use when…             | You already have labeled preference pairs | You need a dynamic, live reward signal          |


In [ ]:
def build_preference_pairs(novels=None, max_chapters=4, max_pairs=30):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "horror", "literary"]

    chapter_files = []
    for alias in novels:  # gather each requested novel's chapter files
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        chapter_files.extend(sorted(novel_path.glob("chapter-*.txt"))[:max_chapters])

    all_paragraphs = []
    for path in chapter_files:  # split every chapter into paragraphs, grouped per chapter
        paras = [
            p.strip().replace("\n", " ")
            for p in path.read_text(encoding="utf-8").split("\n\n")
            if len(p.strip()) > 200
        ]
        all_paragraphs.append(paras)

    pairs = []
    for c_idx, paras in enumerate(all_paragraphs):  # walk each chapter's paragraphs to build pairs
        other_chapter = all_paragraphs[
            (c_idx + 1) % len(all_paragraphs)
        ]  # next chapter (wraps around) supplies the dispreferred/rejected paragraphs
        for i in range(len(paras) - 1):  # pair each paragraph with the real next one (chosen) vs. an unrelated one (rejected)
            prompt = f"{INSTRUCTION_PREFIX}{paras[i]}\n\n"
            chosen = paras[i + 1]  # real next paragraph  -> preferred
            rejected = other_chapter[
                i % len(other_chapter)
            ]  # unrelated paragraph -> dispreferred
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pairs[:max_pairs]


preference_pairs = build_preference_pairs()
print(
    f"Built {len(preference_pairs)} preference pairs  |  "
    f"columns: {list(preference_pairs[0].keys())}"
)

> **PyTorch → Keras:** `from trl import DPOTrainer, DPOConfig` / `DPOTrainer(model=..., ref_model=None, ...)` — TRL's `DPOTrainer` runs the full Direct Preference Optimization loop in PyTorch: `ref_model=None` tells it to auto-create a frozen copy of the current policy as the reference snapshot the KL penalty (`beta`) is measured against, then trains on `(prompt, chosen, rejected)` triples. **Keras/TF equivalent:** none — TRL's trainers (`DPOTrainer`, `PPOTrainer`, etc.) are PyTorch-only with no TensorFlow/Keras backend; a from-scratch Keras DPO implementation would need to manually compute the DPO loss (log-probability margins between a frozen reference `tf.keras.Model` copy and the trainable policy) inside a custom `train_step`.

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_dataset = Dataset.from_list(preference_pairs)  # wrap the preference pairs in a HF Dataset

# Configure a short DPO run (low LR keeps updates small on top of the already-tuned adapter)
dpo_args = DPOConfig(
    output_dir="./checkpoints/preference-dpo",
    per_device_train_batch_size=1,
    max_steps=30,
    learning_rate=1e-5,
    beta=0.1,  # KL penalty -- how tightly the policy stays near the reference snapshot
    bf16=False,  # CPU training: disable bf16 (TRL 1.8+ defaults bf16=True when fp16=False)
    report_to="none",
)

# ref_model=None: DPOTrainer auto-creates a frozen copy of instruct_lora_model as the reference
dpo_trainer = DPOTrainer(
    model=instruct_lora_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

dpo_trainer.train()
instruct_lora_model.save_pretrained("./checkpoints/preference-dpo")  # persist the DPO-aligned adapter weights
print("Saved DPO-aligned adapter.")

In [ ]:
# Quick health check after DPO


# Print one labeled prompt/output pair for a quick qualitative check
def _dpo_test(label, prompt):
    print(f"=== {label} ===")
    print(f"  Input : {repr(prompt)}")
    print(f"  Output: {generate(instruct_lora_model, prompt)}")
    print()


_dpo_test(
    "Test 1: Preferred style (should be more concise than pre-DPO)",
    INSTRUCTION_PREFIX + "Who is Aria Voss?\n\n",
)
_dpo_test(
    "Test 2: Still coherent on domain tasks",
    INSTRUCTION_PREFIX + "Continue: Aria checked the panel and\n\n",
)

print("=== Test 3: Mode-collapse check (outputs should vary across attempts) ===")
for i in range(3):  # run the same prompt 3x to check for repetitive/mode-collapsed outputs
    p = INSTRUCTION_PREFIX + f"Describe the Meridian (attempt {i}).\n\n"
    print(f"  [attempt {i}]  Input : {repr(p[:60])}...")
    print(f"            Output: {generate(instruct_lora_model, p)}")
    print()

---

## End of Part 1: What's Been Trained So Far

Three checkpoints exist on disk now, all saved under `./checkpoints/` (relative to the workspace
root, not this notebook's folder -- see the `CONTENT_DIR` resolution cell near the top for why):

| Checkpoint on disk                   | What it is                                                              |
| ------------------------------------- | ------------------------------------------------------------------------ |
| `./checkpoints/non-instruction-full` | Continued pretraining, full fine-tuning (Concept 1)                     |
| `./checkpoints/instruction-lora`     | Instruction tuning, LoRA adapter (Concept 2)                            |
| `./checkpoints/preference-dpo`       | Preference alignment, DPO on top of the instruction adapter (Concept 3) |

Riverside now has a model that knows its catalog, follows instructions, and has attempted (with mixed
results, honestly reported above) to match editor preference. What's still unanswered: **how many of
the model's weights did each of those stages actually need to update, and is there a cheaper way?**
That's the parameter-based axis -- continue to
**[Part 2: Parameter-Based Techniques + QLoRA & Quantization](02-llm-finetuning-parameter-techniques.ipynb)**,
which reloads these three checkpoints from disk and trains three more (full fine-tuning parameter
count, partial freezing, and LoRA continued pretraining) before introducing QLoRA and a real,
runnable look at post-training quantization.